
# 05. Player-Level Feature Engineering & Combine–Game Integration

## WR-MovementIQ: NFL Big Data Bowl 2027

**Research Project:** Beyond the 40-Yard Dash: Linking NFL Combine Receiver Movement Mechanics to Regular-Season Performance Using Player Tracking Data.

**Notebook:** 05 — Player-Level Feature Engineering & Combine–Game Integration

**Competition:** NFL Big Data Bowl 2027 — Open Track

**Study population:** Wide receivers

**Draft cohorts:** 2023, 2024 and 2025

**Environment:** Python / Google Colab

---

### Notebook purpose

This notebook transforms raw Combine and regular-season tracking observations into player-level analytical features.

It develops Combine movement profiles, regular-season player-season movement profiles and an integrated dataset linking pre-draft movement characteristics to post-draft NFL performance.

The resulting datasets provide the analytical foundation for investigating whether movement characteristics measured during NFL Combine drills are associated with movement characteristics observed during regular-season games.



## 1. Research Objectives

The primary objective is to construct a reliable player-level dataset linking NFL Combine movement characteristics with regular-season receiver movement characteristics.

Specific objectives:

1. Aggregate Combine drill-level movement features into player-level profiles.
2. Aggregate regular-season receiver tracking observations into player-season movement profiles.
3. Link Combine participants with their subsequent regular-season tracking records.
4. Preserve draft-year and season information to distinguish pre-draft from post-draft observations.
5. Integrate traditional Combine athletic measurements with tracking-derived movement features.
6. Evaluate player exposure and data completeness.
7. Assess predictor availability and correlation before modelling.
8. Construct reproducible analytical samples for subsequent predictive experiments.

### Main research question

To what extent do movement characteristics measured during NFL Combine receiver-specific drills explain variation in wide receivers' in-game movement characteristics during regular-season NFL plays?

### Analytical principle

Combine characteristics are treated as pre-draft explanatory variables, while regular-season player-season movement characteristics are treated as outcome variables.



## 2. Data Sources

The notebook uses the following project datasets.

### Combine data

- `combine_results.csv`
- `wr_combine_attempt_features_enriched.parquet`
- `wr_combine_turning_candidate_features.parquet`

### Regular-season tracking data

- `wr_game_movement_features.parquet`
- `player_play.csv`
- `players.csv`
- `games.csv`

### Supporting player information

- `player_career_successes.csv`

### Previously engineered datasets

The notebook uses attempt-level Combine features and receiver-play-level game movement features produced during earlier data preparation and feature-engineering stages.

### Data organization

All analytical datasets are maintained within:

`data/features/`

Raw source datasets remain in:

`data/raw/nfl-big-data-bowl-2027/`

Intermediate audits and diagnostic reports are maintained in:

`reports/data_audit/`



## 3. Combine Player-Level Feature Engineering

The Combine tracking dataset contains multiple observations for each player across different drills and attempts.

To construct player-level explanatory variables, attempt-level movement features are aggregated first by player and drill, then across the selected primary drills.

### Primary receiver skill drills

The following eleven drills are used:

1. GAUNTLET_DRILL
2. DAGGER_ROUTE_LEFT
3. SLANT_ROUTE_LEFT
4. CURL_ROUTE_RIGHT
5. SLOT_STRIKE_ROUTE_LEFT
6. OVER_SHOULDER_ADJUST
7. RED_ZONE_FADE_RIGHT
8. COMEBACK_ROUTE_RIGHT
9. SLOT_SAIL_ROUTE_RIGHT
10. GO_ROUTE_RIGHT
11. POST_CORNER_ROUTE_RIGHT

The 40-yard dash is retained separately as a traditional athletic benchmark.

### Movement characteristics

The engineered features include:

- Mean speed.
- Peak speed.
- 95th percentile speed.
- Mean acceleration.
- Peak acceleration.
- Coordinate-derived path length.
- Net displacement.
- Path efficiency.
- Total turning.
- Mean turning.
- 95th percentile turning.
- Turning rate.
- Turning-pair coverage.

### Aggregation strategy

For each player, movement characteristics are summarized across available primary drills using:

- Median across drills.
- Interquartile range across drills.

The median describes typical movement characteristics across the observed drills, while the interquartile range describes variability.

### Important consideration

Players do not necessarily have complete observations for all eleven drills. Drill coverage and the number of observed attempts are retained as supporting quality indicators.



## 4. Combine Feature Engineering Outputs

The Combine feature-engineering process produced two principal player-level datasets.

### 4.1 Player-drill profiles

`wr_combine_player_drill_profiles.parquet`

`wr_combine_player_drill_profiles.csv`

These datasets summarize movement characteristics at the player-and-drill level.

### 4.2 Player-level Combine profiles

`wr_combine_player_profiles.parquet`

`wr_combine_player_profiles.csv`

These datasets summarize movement characteristics across the selected primary drills.

### Structural validation

The player-level Combine profile contains:

- 107 unique wide receivers.
- 30 columns.
- Complete coverage of the linked Combine player population.

The profile provides the principal source of tracking-derived Combine explanatory variables for the integrated analytical dataset.

### Interpretation

These variables describe observed movement characteristics during Combine drills.

They should not automatically be interpreted as direct measurements of agility, route-running ability or future NFL performance.



## 5. Regular-Season Player-Season Feature Engineering

Receiver-play-level tracking observations are aggregated into player-season profiles.

The analysis focuses on receiver movement between the ball-snap event and the pass-forward event.

This window represents pre-throw receiver movement rather than the complete route.

### Game movement characteristics

The engineered outcome variables include:

- Median and mean game speed.
- Median and mean peak speed.
- Median and mean acceleration.
- Median and mean total direction change.
- Median and mean path efficiency.
- Median and mean separation at pass forward.
- Median and mean receiving yards, where available.

### Supporting exposure variables

The player-season profile also includes:

- Number of receiver plays.
- Number of games.
- Number of observed route types.
- Tracking observations.
- Plays with receiving-yard information.
- Plays with separation information.
- Receiving-yard coverage percentage.
- Separation coverage percentage.
- Position classification.

### Aggregation unit

The analytical unit is:

**One player in one regular season.**

Consequently, a player observed across three seasons contributes three player-season records.

These observations are not independent players. Player identifiers must therefore be respected during subsequent model validation.



## 6. Game Feature Engineering Outputs

The player-season aggregation produced:

`wr_game_player_season_profiles.parquet`

`wr_game_player_season_profiles.csv`

### Structural validation

| Metric | Result |
|---|---:|
| Player-season observations | 146 |
| Unique players | 79 |
| Draft cohorts | 3 |
| Seasons | 2023, 2024, 2025 |
| Feature columns | 41 |

### Season distribution

The dataset includes player-season observations from all three available regular seasons.

### Exposure considerations

Players have different numbers of observed receiver plays and games.

This difference matters because a player with only a few observed plays may have less stable season-level movement estimates than a player with extensive exposure.

The number of receiver plays is therefore retained as an important analytical exposure variable.



## 7. Combine–Game Player Integration

The next stage links each player's pre-draft Combine profile with their post-draft regular-season movement profile.

### Player linkage

The integration uses the NFL player identifier:

`nfl_id`

Draft-year information is retained to identify the relevant Combine cohort.

### Temporal validity

Only regular-season observations occurring in the player's draft year or later are eligible for the integrated analytical dataset.

Pre-draft game records are excluded.

### Linkage results

| Metric | Result |
|---|---:|
| Combine wide receivers | 107 |
| Matched players | 79 |
| Unmatched players | 28 |
| Integrated player-season records | 146 |

The 28 unmatched players are retained in the Combine descriptive population but cannot contribute to the integrated Combine–game analysis using the available tracking records.

### Important distinction

An unmatched player is not necessarily an unsuccessful player.

The absence of a matching record indicates only that a corresponding regular-season tracking observation was not available in the datasets used for this study.



## 8. Integrated Dataset Validation

The integrated dataset combines:

- Player-level Combine movement characteristics.
- Regular-season player-season movement characteristics.
- Draft-year information.
- Season information.
- Player exposure information.
- Position classification.

### Integrated dataset

`wr_combine_game_integrated.parquet`

`wr_combine_game_integrated.csv`

### Structural validation

| Validation | Result |
|---|---:|
| Integrated observations | 146 |
| Unique players | 79 |
| Duplicate player-season records | 0 |
| Invalid pre-draft records | 0 |
| Missing position classifications | 0 |

### Position distribution

The integrated dataset includes:

- WR.
- SLOT_WR.

### Temporal validation

The integration confirms that all included player-season records occur in the player's draft year or a subsequent season.

This ensures that Combine characteristics are temporally prior to the corresponding NFL regular-season outcomes.



## 9. Integration of Traditional Combine Measurements

Traditional Combine athletic measurements are integrated into the player-season dataset.

These measurements provide a benchmark against which tracking-derived Combine features can be evaluated.

### Traditional measurements

- 10-yard split.
- 40-yard dash.
- Vertical jump.
- Broad jump.
- Three-cone drill.
- Short shuttle.
- Bench press repetitions.
- NGS athleticism score.
- NGS college production score.
- NGS final score.

### Merge validation

The traditional Combine data contained:

- 107 wide receiver records.
- 107 unique player identifiers.
- No duplicate player-draft keys.

The merge successfully matched all 146 integrated player-season records.

### Variable coverage

| Measurement | Coverage |
|---|---:|
| NGS final score | 100.00% |
| NGS college production score | 100.00% |
| NGS athleticism score | 100.00% |
| 10-yard split | 89.73% |
| 40-yard dash | 89.73% |
| Broad jump | 89.04% |
| Vertical jump | 87.67% |
| Short shuttle | 39.73% |
| Three-cone drill | 32.88% |
| Bench press repetitions | 26.03% |

### Interpretation

Traditional Combine measurements have different levels of availability.

The lower coverage of the three-cone drill, short shuttle and bench press must be considered when constructing traditional-only models.

The NGS scores provide complete coverage in this integrated sample but should not be treated as interchangeable with individual physical measurements.



## 10. Player Exposure and Analytical Sample Selection

Player-season records differ in the number of receiver plays observed.

Low exposure may produce unstable season-level movement estimates.

An exposure sensitivity analysis was therefore conducted.

### Exposure sensitivity

| Minimum receiver plays | Player-seasons | Unique players |
|---|---:|---:|
| 1 | 146 | 79 |
| 5 | 129 | 70 |
| 10 | 116 | 66 |
| 20 | 93 | 54 |
| 30 | 80 | 45 |
| 50 | 54 | 31 |

### Primary modelling sample

A minimum of ten receiver plays per player-season was selected for the primary modelling sample.

This produced:

- 116 player-season observations.
- 66 unique players.
- Three draft cohorts.

### Sensitivity sample

A stricter threshold of twenty receiver plays produced:

- 93 player-season observations.
- 54 unique players.
- Three draft cohorts.

### Robustness sample

The complete 146-record dataset is retained for analyses that do not require an exposure restriction.

### Validation principle

All future model validation must group observations by `nfl_id`.

Player-season records from the same player must never appear in both training and testing partitions.



## 11. Outcome and Predictor Readiness Assessment

Before modelling, candidate predictors and outcomes were evaluated for completeness and statistical suitability.

### Candidate Combine predictors

Two predictor groups were defined.

#### Model A — Traditional Combine predictors

- Forty-yard dash.
- Vertical jump.
- Broad jump.

#### Model B — Tracking-derived Combine predictors

- Combine median mean speed.
- Combine median peak speed.
- Combine median mean acceleration.
- Combine median path efficiency.
- Combine median coordinate total turn.
- Combine median coordinate 95th percentile turn.

#### Model C — Combined predictors

Model A and Model B features are combined.

### Candidate game outcomes

The following player-season characteristics were considered:

- Median game mean speed.
- Median game peak speed.
- Median game mean acceleration.
- Median game total direction change.
- Median game path efficiency.
- Median game separation at pass forward.

### Outcome availability

The selected movement outcomes were complete in the primary integrated sample.

Receiving-yard outcomes had a separate missingness issue at play level, although player-season median receiving yards were available for 145 of 146 records.

### Measurement qualification

Game tracking speed and acceleration variables retain their `_raw` suffix because their physical units have not yet been independently established.

Their interpretation must remain consistent with this limitation.



## 12. Predictor Correlation and Exploratory Association Analysis

Correlation analysis was conducted to evaluate relationships among candidate predictors and between Combine predictors and game movement outcomes.

Spearman rank correlation was used to assess monotonic associations.

### Traditional predictor relationships

A relatively strong positive association was observed between:

- NGS college production score.
- NGS final score.

Their Spearman correlation was approximately 0.80.

This suggests that these two measures contain overlapping information.

### Tracking-derived predictor relationships

No pair of selected tracking-derived predictors exhibited a Spearman correlation of 0.80 or higher.

Some moderate relationships were observed among speed, acceleration and turning characteristics.

### Exploratory predictor-outcome relationships

Several preliminary associations were observed, including:

- Forty-yard dash and game peak speed.
- Combine tracking peak speed and game peak speed.
- Combine path efficiency and game path efficiency.
- Combine turning characteristics and game acceleration.

These relationships are exploratory.

They do not establish predictive performance, statistical significance or causal effects.

### Output files

- `traditional_predictor_spearman_correlations.csv`
- `tracking_predictor_spearman_correlations.csv`
- `outcome_predictor_spearman_correlations.csv`



## 13. Final Feature Engineering Outputs

The following datasets were generated during this notebook.

### Combine player-level features

`data/features/wr_combine_player_drill_profiles.parquet`

`data/features/wr_combine_player_drill_profiles.csv`

`data/features/wr_combine_player_profiles.parquet`

`data/features/wr_combine_player_profiles.csv`

### Game player-season features

`data/features/wr_game_player_season_profiles.parquet`

`data/features/wr_game_player_season_profiles.csv`

### Integrated analytical dataset

`data/features/wr_combine_game_integrated.parquet`

`data/features/wr_combine_game_integrated.csv`

### Integrated dataset with traditional Combine variables

`data/features/wr_combine_game_integrated_v2.parquet`

`data/features/wr_combine_game_integrated_v2.csv`

### Modelling samples

`data/features/wr_movementiq_primary_sample.parquet`

`data/features/wr_movementiq_primary_sample.csv`

`data/features/wr_movementiq_sensitivity_20plus.parquet`

`data/features/wr_movementiq_sensitivity_20plus.csv`

### Audit reports

Saved under:

`reports/data_audit/`

These include traditional Combine coverage, predictor correlations, outcome readiness and exposure sensitivity results.



## 14. Methodological Decisions

The following decisions were established during feature engineering and data integration.

| Area | Decision |
|---|---|
| Combine population | 107 wide receivers |
| Integrated population | 79 matched players |
| Analytical unit | Player-season |
| Primary exposure threshold | At least 10 receiver plays |
| Sensitivity threshold | At least 20 receiver plays |
| Robustness sample | All available player-seasons |
| Combine predictors | Traditional and tracking-derived features |
| Primary game outcomes | Six movement characteristics |
| Validation grouping | `nfl_id` |
| Temporal restriction | Draft year or later |
| Receiving yards | Supporting outcome due to play-level missingness |
| Speed and acceleration | Retain raw-unit qualification |

### Important methodological considerations

1. Multiple player-season observations from one player are not independent.
2. Players with low receiver-play exposure may have unstable movement estimates.
3. Different draft cohorts have unequal observation windows.
4. Traditional Combine measurements have different missingness levels.
5. Game movement variables represent the snap-to-throw window, not the entire receiver route.
6. The analysis is observational and cannot establish causal effects.



## 15. Limitations

The integrated dataset has several limitations.

### Sample size

The integrated analysis includes 79 matched players, with 66 players in the primary exposure-restricted sample.

This limits the complexity of statistical models that can be supported reliably.

### Unequal player exposure

Players have different numbers of receiver plays and games.

This may affect the stability of player-season movement estimates.

### Unequal cohort observation windows

The 2025 draft cohort has fewer observed post-draft seasons than the earlier cohorts.

Consequently, long-term career comparisons cannot be made using the current data.

### Missing traditional measurements

Some traditional Combine measurements have relatively low coverage.

Model comparisons must account for the available sample and avoid silently changing the population between models.

### Measurement interpretation

Game tracking movement variables retain raw-unit labels where units have not been independently verified.

### Observational research design

Associations between Combine measurements and NFL game movement characteristics cannot be interpreted as causal effects.



## 16. Conclusion and Transition to Measurement Validation

This notebook established the player-level analytical datasets required for the WR-MovementIQ research project.

Combine attempt-level movement characteristics were aggregated into player-level profiles.

Regular-season receiver tracking observations were aggregated into player-season movement profiles.

The two datasets were integrated using player identifiers and temporal restrictions to ensure that Combine measurements preceded the corresponding NFL observations.

The resulting integrated dataset contains 146 player-season observations from 79 matched players.

The primary modelling sample contains 116 player-season observations from 66 players after applying the minimum ten-receiver-play exposure threshold.

Traditional Combine measurements were also integrated, and predictor availability and correlation were evaluated.

### Transition to Notebook 06

Before interpreting turning characteristics in predictive models, the reliability and comparability of directional measurements must be examined.

Notebook 06 therefore investigates:

- Coordinate-derived heading validation.
- Turning reconstruction.
- Speed-threshold sensitivity.
- Directional measurement error.
- Cohort and drill comparability.
- Speed-composition standardization.

### Research principle

A valid integrated dataset is necessary, but it is not sufficient. The measurement properties of its explanatory variables must also be established before drawing substantive conclusions.


In [2]:
import os
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/NFL-BigDataBowl-2027"
)

print("DRIVE MOUNTED:", os.path.ismount("/content/drive"))
print("PROJECT EXISTS:", PROJECT.exists())

if PROJECT.exists():

    print("\nPROJECT DIRECTORIES")

    for folder in ["data", "data/features", "data/raw",
                   "notebooks", "reports/data_audit"]:

        path = PROJECT / folder
        print(folder, ":", path.exists())

    feature_dir = PROJECT / "data/features"

    if feature_dir.exists():

        print("\nFEATURE FILES")

        for file in sorted(feature_dir.iterdir()):
            print(file.name)

else:
    print("\nPROJECT ROOT NOT FOUND")
    print("Check whether Google Drive needs remounting.")

DRIVE MOUNTED: False
PROJECT EXISTS: True

PROJECT DIRECTORIES
data : False
data/features : False
data/raw : False
notebooks : False
reports/data_audit : True


In [4]:
import os

for path in [
    "/content/drive",
    "/content/drive/MyDrive",
    "/content/gdrive",
    "/content/gdrive2"
]:
    print(f"\n{path}")
    print("Exists:", os.path.exists(path))
    print("Is mount:", os.path.ismount(path))

    if os.path.isdir(path):
        print("Contents:", os.listdir(path)[:15])


/content/drive
Exists: True
Is mount: False
Contents: ['MyDrive']

/content/drive/MyDrive
Exists: True
Is mount: False
Contents: ['NFL-BigDataBowl-2027']

/content/gdrive
Exists: False
Is mount: False

/content/gdrive2
Exists: False
Is mount: False


In [5]:
from google.colab import drive
drive.mount("/content/gdrive_new")

Mounted at /content/gdrive_new


In [6]:
from pathlib import Path

search_roots = [
    Path("/content/drive"),
    Path("/content/gdrive_new")
]

for base in search_roots:
    if base.exists():
        print("\nSEARCHING:", base)

        for path in base.glob("**/NFL-BigDataBowl-2027"):
            print("PROJECT FOUND:", path)


SEARCHING: /content/drive
PROJECT FOUND: /content/drive/MyDrive/NFL-BigDataBowl-2027

SEARCHING: /content/gdrive_new
PROJECT FOUND: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027
PROJECT FOUND: /content/gdrive_new/.Encrypted/MyDrive/NFL-BigDataBowl-2027


In [7]:
from pathlib import Path

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data/features"

print("FEATURE DIRECTORY EXISTS:", FEATURES.exists())

required_files = [
    "wr_combine_attempt_features_enriched.parquet",
    "wr_combine_attempt_features_v2.parquet",
    "wr_combine_turning_candidate_features.parquet",
    "wr_game_movement_features.parquet"
]

for filename in required_files:
    path = FEATURES / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

FEATURE DIRECTORY EXISTS: True
wr_combine_attempt_features_enriched.parquet: FOUND
wr_combine_attempt_features_v2.parquet: FOUND
wr_combine_turning_candidate_features.parquet: FOUND
wr_game_movement_features.parquet: FOUND


In [8]:
ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

In [9]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data/features"
AUDIT = ROOT / "reports/data_audit"

combine = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

game = pd.read_parquet(
    FEATURES / "wr_game_movement_features.parquet"
)

print("ENRICHED COMBINE DATA")
print("Rows:", len(combine))
print("Players:", combine["nfl_id"].nunique())
print("Unique attempts:", combine.duplicated(
    ["draft_year", "nfl_id", "drill_type",
     "drill_name", "attempt", "event_id"]
).sum() == 0)

print("\nGAME MOVEMENT DATA")
print("Rows:", len(game))
print("Players:", game["nfl_id"].nunique())

print("\nPROJECT:", ROOT)

ENRICHED COMBINE DATA
Rows: 1771
Players: 107
Unique attempts: True

GAME MOVEMENT DATA
Rows: 6801
Players: 107

PROJECT: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027


In [10]:
ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

In [11]:

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data/features"
AUDIT = ROOT / "reports/data_audit"

AUDIT.mkdir(parents=True, exist_ok=True)

combine = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

game = pd.read_parquet(
    FEATURES / "wr_game_movement_features.parquet"
)

combine_players = (
    combine[["nfl_id", "draft_year"]]
    .drop_duplicates()
)

assert not combine_players["nfl_id"].duplicated().any()

# 1. Drill coverage
total_players = combine_players["nfl_id"].nunique()

drill_coverage = (
    combine.groupby(["drill_name", "drill_type"])
    .agg(
        attempts=("nfl_id", "size"),
        players=("nfl_id", "nunique"),
        median_attempts_per_player=(
            "nfl_id", lambda x: x.value_counts().median()
        ),
        max_attempts_per_player=(
            "nfl_id", lambda x: x.value_counts().max()
        )
    )
    .reset_index()
)

drill_coverage["player_coverage_pct"] = (
    drill_coverage["players"] / total_players * 100
)

drill_coverage = drill_coverage.sort_values(
    ["players", "attempts"], ascending=False
)

print("COMBINE DRILL COVERAGE")
display(drill_coverage.round(2))

# 2. Post-draft game observations
game_cohort = game.merge(
    combine_players,
    on="nfl_id",
    how="inner",
    validate="many_to_one"
)

game_post = game_cohort.loc[
    (game_cohort["season"] >= game_cohort["draft_year"])
    & game_cohort["position"].isin(["WR", "SLOT_WR"])
].copy()

player_season = (
    game_post.groupby(["draft_year", "season", "nfl_id"])
    .agg(
        receiver_plays=("play_id", "size"),
        games=("game_id", "nunique"),
        median_tracking_observations=(
            "tracking_observations", "median"
        )
    )
    .reset_index()
)

print("\nPLAYER-SEASON OBSERVATIONS:", len(player_season))
print("UNIQUE PLAYERS:", player_season["nfl_id"].nunique())

# 3. Exposure sensitivity
threshold_rows = []

for threshold in [1, 5, 10, 20, 30, 50]:
    eligible = player_season[
        player_season["receiver_plays"] >= threshold
    ]

    threshold_rows.append({
        "minimum_receiver_plays": threshold,
        "player_seasons": len(eligible),
        "unique_players": eligible["nfl_id"].nunique(),
        "draft_cohorts": eligible["draft_year"].nunique(),
        "median_plays": eligible["receiver_plays"].median()
    })

exposure_sensitivity = pd.DataFrame(threshold_rows)

print("\nEXPOSURE SENSITIVITY")
display(exposure_sensitivity)

# 4. Exposure by draft cohort
cohort_exposure = (
    player_season.groupby("draft_year")
    .agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique"),
        median_plays=("receiver_plays", "median"),
        mean_plays=("receiver_plays", "mean"),
        min_plays=("receiver_plays", "min"),
        max_plays=("receiver_plays", "max"),
        median_games=("games", "median"),
        min_games=("games", "min"),
        max_games=("games", "max")
    )
    .reset_index()
)

print("\nEXPOSURE BY DRAFT COHORT")
display(cohort_exposure.round(2))

# 5. Save
player_season.to_parquet(
    FEATURES / "wr_player_season_exposure.parquet",
    index=False
)

player_season.to_csv(
    FEATURES / "wr_player_season_exposure.csv",
    index=False
)

drill_coverage.to_csv(
    AUDIT / "combine_drill_player_coverage.csv",
    index=False
)

exposure_sensitivity.to_csv(
    AUDIT / "game_exposure_threshold_sensitivity.csv",
    index=False
)

cohort_exposure.to_csv(
    AUDIT / "game_exposure_by_draft_cohort.csv",
    index=False
)

print("\nALL EXPOSURE AUDITS SAVED SUCCESSFULLY.")


COMBINE DRILL COVERAGE


,drill_name,drill_type,attempts,players,median_attempts_per_player,max_attempts_per_player,player_coverage_pct
5,GAUNTLET_DRILL,SKILL_DRILLS_WR,200,107,2.0,2,100.00
3,DAGGER_ROUTE_LEFT,SKILL_DRILLS_WR,115,107,1.0,2,100.00
12,SLANT_ROUTE_LEFT,SKILL_DRILLS_WR,110,107,1.0,2,100.00
2,CURL_ROUTE_RIGHT,SKILL_DRILLS_WR,110,103,1.0,2,96.26
14,SLOT_STRIKE_ROUTE_LEFT,SKILL_DRILLS_WR,102,101,1.0,2,94.39
7,OVER_SHOULDER_ADJUST,SKILL_DRILLS_WR,101,100,1.0,2,93.46
10,RED_ZONE_FADE_RIGHT,SKILL_DRILLS_WR,112,98,1.0,3,91.59
1,COMEBACK_ROUTE_RIGHT,SKILL_DRILLS_WR,110,98,1.0,2,91.59
13,SLOT_SAIL_ROUTE_RIGHT,SKILL_DRILLS_WR,106,98,1.0,2,91.59
6,GO_ROUTE_RIGHT,SKILL_DRILLS_WR,105,98,1.0,2,91.59



PLAYER-SEASON OBSERVATIONS: 146
UNIQUE PLAYERS: 79

EXPOSURE SENSITIVITY


,minimum_receiver_plays,player_seasons,unique_players,draft_cohorts,median_plays
0,1,146,79,3,33.0
1,5,129,70,3,42.0
2,10,116,66,3,46.0
3,20,93,54,3,56.0
4,30,80,45,3,60.5
5,50,54,31,3,78.0



EXPOSURE BY DRAFT COHORT


,draft_year,player_seasons,unique_players,median_plays,mean_plays,min_plays,max_plays,median_games,min_games,max_games
0,2023,80,31,34.5,45.72,1,156,12.0,1,17
1,2024,45,27,37.0,41.58,1,132,11.0,1,17
2,2025,21,21,29.0,31.57,3,87,12.0,1,17



ALL EXPOSURE AUDITS SAVED SUCCESSFULLY.


In [12]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data/features"

combine = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

game = pd.read_parquet(
    FEATURES / "wr_game_movement_features.parquet"
)

print("=" * 75)
print("COMBINE ENRICHED FEATURE COLUMNS")
print("=" * 75)

for col in combine.columns:
    print(col)

print("\n" + "=" * 75)
print("GAME MOVEMENT FEATURE COLUMNS")
print("=" * 75)

for col in game.columns:
    print(col)

print("\n" + "=" * 75)
print("COMBINE DRILL TYPES")
print("=" * 75)

print(
    combine[["drill_name", "drill_type"]]
    .drop_duplicates()
    .sort_values("drill_name")
    .to_string(index=False)
)

print("\nSCHEMA INSPECTION COMPLETE")


COMBINE ENRICHED FEATURE COLUMNS
draft_year
nfl_id
event_id
drill_type
drill_name
attempt
observations
start_time
end_time
duration_seconds
start_x
start_y
end_x
end_y
mean_speed
peak_speed
p95_speed
speed_std
mean_acceleration
peak_acceleration
minimum_acceleration
acceleration_std
coordinate_path_length
summed_recorded_displacement
total_direction_change
median_sampling_interval
maximum_sampling_interval
large_sampling_gaps
net_displacement
path_efficiency
valid_turn_pairs
coordinate_total_turn
coordinate_mean_turn
coordinate_median_turn
coordinate_p95_turn
coordinate_max_turn
recorded_total_turn
recorded_mean_turn
speed_threshold
turning_rate_degrees_per_second
valid_pair_coverage
turning_coverage_flag

GAME MOVEMENT FEATURE COLUMNS
season
game_id
play_id
nfl_id
position
route_ran
separation_at_pass_forward
receiving_yards
snap_to_throw_seconds
tracking_observations
mean_speed_raw
peak_speed_raw
p95_speed_raw
mean_acceleration_raw
peak_acceleration_raw
minimum_acceleration_raw
summe

In [13]:

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data/features"
AUDIT = ROOT / "reports/data_audit"

FEATURES.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

combine = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

# -------------------------------------------------------
# 1. DEFINE PRIMARY SKILL DRILLS
# -------------------------------------------------------

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

# Attempt-level features available in the enriched dataset.
MOVEMENT_FEATURES = [
    "duration_seconds",
    "mean_speed",
    "peak_speed",
    "p95_speed",
    "speed_std",
    "mean_acceleration",
    "peak_acceleration",
    "minimum_acceleration",
    "coordinate_path_length",
    "net_displacement",
    "path_efficiency",
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "turning_rate_degrees_per_second"
]

QUALITY_FEATURES = [
    "observations",
    "valid_turn_pairs",
    "valid_pair_coverage",
    "large_sampling_gaps"
]

# -------------------------------------------------------
# 2. CHECK DRILL AVAILABILITY
# -------------------------------------------------------

available_drills = set(combine["drill_name"].dropna().unique())

missing_drills = sorted(
    set(PRIMARY_DRILLS) - available_drills
)

assert not missing_drills, (
    f"Expected primary drills missing: {missing_drills}"
)

primary_attempts = combine[
    combine["drill_name"].isin(PRIMARY_DRILLS)
].copy()

print("PRIMARY SKILL-DRILL ATTEMPTS:", len(primary_attempts))
print("PLAYERS:", primary_attempts["nfl_id"].nunique())
print("DRILLS:", primary_attempts["drill_name"].nunique())

# -------------------------------------------------------
# 3. AGGREGATE REPEATED ATTEMPTS BY PLAYER AND DRILL
# -------------------------------------------------------

group_keys = [
    "draft_year",
    "nfl_id",
    "drill_name"
]

aggregation = {
    feature: ["median", "std"]
    for feature in MOVEMENT_FEATURES
}

aggregation.update({
    feature: ["median"]
    for feature in QUALITY_FEATURES
})

player_drill = (
    primary_attempts
    .groupby(group_keys)
    .agg(aggregation)
)

# Flatten multi-level columns.
player_drill.columns = [
    f"{feature}_{stat}"
    for feature, stat in player_drill.columns
]

player_drill = player_drill.reset_index()

attempt_counts = (
    primary_attempts
    .groupby(group_keys)
    .size()
    .rename("attempt_count")
    .reset_index()
)

player_drill = player_drill.merge(
    attempt_counts,
    on=group_keys,
    how="left",
    validate="one_to_one"
)

print("\nPLAYER-DRILL DATASET")
print("Rows:", len(player_drill))
print("Players:", player_drill["nfl_id"].nunique())
print("Unique player-drill combinations:",
      player_drill.duplicated(group_keys).sum() == 0)

# -------------------------------------------------------
# 4. CREATE COMPACT PLAYER-LEVEL COMBINE PROFILES
# -------------------------------------------------------

# These are the primary movement dimensions.
PROFILE_FEATURES = [
    "mean_speed",
    "peak_speed",
    "p95_speed",
    "mean_acceleration",
    "peak_acceleration",
    "coordinate_path_length",
    "net_displacement",
    "path_efficiency",
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "turning_rate_degrees_per_second"
]

profile_rows = []

for (draft_year, nfl_id), group in player_drill.groupby(
    ["draft_year", "nfl_id"]
):

    row = {
        "draft_year": draft_year,
        "nfl_id": nfl_id,
        "primary_drills_observed": group["drill_name"].nunique(),
        "primary_attempts_observed": group["attempt_count"].sum()
    }

    for feature in PROFILE_FEATURES:

        col = f"{feature}_median"

        values = pd.to_numeric(
            group[col], errors="coerce"
        ).dropna()

        row[f"combine_{feature}_median_across_drills"] = (
            values.median() if len(values) else np.nan
        )

        row[f"combine_{feature}_iqr_across_drills"] = (
            values.quantile(0.75) - values.quantile(0.25)
            if len(values) >= 2 else np.nan
        )

    # Quality diagnostic, not a model predictor.
    coverage = pd.to_numeric(
        group["valid_pair_coverage_median"],
        errors="coerce"
    )

    row["median_turning_pair_coverage"] = coverage.median()
    row["drills_with_turning_coverage_below_70_pct"] = (
        coverage.lt(0.70).sum()
    )

    profile_rows.append(row)

combine_player_profile = pd.DataFrame(profile_rows)

# -------------------------------------------------------
# 5. VALIDATE AND SAVE
# -------------------------------------------------------

assert combine_player_profile["nfl_id"].is_unique
assert len(combine_player_profile) == 107

player_drill.to_parquet(
    FEATURES / "wr_combine_player_drill_profiles.parquet",
    index=False
)

player_drill.to_csv(
    FEATURES / "wr_combine_player_drill_profiles.csv",
    index=False
)

combine_player_profile.to_parquet(
    FEATURES / "wr_combine_player_profiles.parquet",
    index=False
)

combine_player_profile.to_csv(
    FEATURES / "wr_combine_player_profiles.csv",
    index=False
)

print("\nCOMBINE PLAYER PROFILE")
print("Rows:", len(combine_player_profile))
print("Players:", combine_player_profile["nfl_id"].nunique())
print("Columns:", len(combine_player_profile.columns))

display(combine_player_profile.head())

print("\nSAVED COMBINE PLAYER DATASETS")
print(FEATURES / "wr_combine_player_drill_profiles.parquet")
print(FEATURES / "wr_combine_player_profiles.parquet")


PRIMARY SKILL-DRILL ATTEMPTS: 1282
PLAYERS: 107
DRILLS: 11

PLAYER-DRILL DATASET
Rows: 1113
Players: 107
Unique player-drill combinations: True

COMBINE PLAYER PROFILE
Rows: 107
Players: 107
Columns: 30


,draft_year,nfl_id,primary_drills_observed,primary_attempts_observed,combine_mean_speed_median_across_drills,combine_mean_speed_iqr_across_drills,combine_peak_speed_median_across_drills,combine_peak_speed_iqr_across_drills,combine_p95_speed_median_across_drills,combine_p95_speed_iqr_across_drills,...,combine_coordinate_total_turn_median_across_drills,combine_coordinate_total_turn_iqr_across_drills,combine_coordinate_mean_turn_median_across_drills,combine_coordinate_mean_turn_iqr_across_drills,combine_coordinate_p95_turn_median_across_drills,combine_coordinate_p95_turn_iqr_across_drills,combine_turning_rate_degrees_per_second_median_across_drills,combine_turning_rate_degrees_per_second_iqr_across_drills,median_turning_pair_coverage,drills_with_turning_coverage_below_70_pct
0,2023,55884,5,6,6.489167,1.089790,8.533333,0.265000,8.520833,0.307000,...,181.390245,95.098824,2.264282,0.947004,6.966447,3.846913,21.619604,11.007591,0.954801,0
1,2023,55885,10,11,5.951118,2.040763,8.698333,0.580833,8.580250,0.703458,...,163.529808,57.738491,3.012131,0.844059,8.166874,3.675739,27.447157,7.465742,0.922161,0
2,2023,55887,5,6,6.080678,1.346808,8.416667,0.350000,8.207333,0.359333,...,162.730758,102.127300,2.506501,0.206404,6.896084,1.994038,23.572984,5.931670,0.953125,0
3,2023,55904,11,12,5.863228,1.911702,8.083333,0.856667,8.025333,0.837250,...,177.740645,65.661474,3.259495,1.307315,8.042895,4.348610,28.973291,9.701266,0.944618,0
4,2023,55915,11,12,5.389407,1.954449,8.680000,1.088333,8.545833,1.103750,...,188.076179,74.874589,2.699430,1.261239,7.304170,6.214443,25.076824,11.049958,0.935755,0



SAVED COMBINE PLAYER DATASETS
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_player_drill_profiles.parquet
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_player_profiles.parquet


In [14]:

# -------------------------------------------------------
# 6. LOAD GAME MOVEMENT DATA
# -------------------------------------------------------

game = pd.read_parquet(
    FEATURES / "wr_game_movement_features.parquet"
)

combine_players = (
    combine_player_profile[
        ["draft_year", "nfl_id"]
    ]
    .drop_duplicates()
)

# Attach Combine draft cohort.
game_cohort = game.merge(
    combine_players,
    on="nfl_id",
    how="inner",
    validate="many_to_one"
)

# Retain post-draft seasons and receiver positions.
game_post = game_cohort.loc[
    (game_cohort["season"] >= game_cohort["draft_year"])
    & game_cohort["position"].isin(["WR", "SLOT_WR"])
].copy()

# Convert numeric-like fields safely.
numeric_candidates = [
    "mean_speed_raw",
    "peak_speed_raw",
    "p95_speed_raw",
    "mean_acceleration_raw",
    "peak_acceleration_raw",
    "minimum_acceleration_raw",
    "summed_recorded_displacement_raw",
    "coordinate_path_length_raw",
    "total_direction_change_raw",
    "path_efficiency",
    "snap_to_throw_seconds",
    "separation_at_pass_forward",
    "receiving_yards",
    "maximum_sampling_gap_seconds",
    "gaps_over_0_15_seconds",
    "rows_with_missing_movement_values",
    "tracking_observations"
]

for col in numeric_candidates:
    if col in game_post.columns:
        game_post[col] = pd.to_numeric(
            game_post[col], errors="coerce"
        )

# -------------------------------------------------------
# 7. DEFINE PLAYER-SEASON AGGREGATIONS
# -------------------------------------------------------

GAME_MOVEMENT_FEATURES = [
    "mean_speed_raw",
    "peak_speed_raw",
    "p95_speed_raw",
    "mean_acceleration_raw",
    "peak_acceleration_raw",
    "minimum_acceleration_raw",
    "summed_recorded_displacement_raw",
    "coordinate_path_length_raw",
    "total_direction_change_raw",
    "path_efficiency",
    "snap_to_throw_seconds"
]

# Median and mean are retained separately.
game_aggregation = {}

for feature in GAME_MOVEMENT_FEATURES:
    game_aggregation[feature] = ["median", "mean"]

# Secondary/contextual variables.
game_aggregation.update({
    "separation_at_pass_forward": ["median", "mean"],
    "receiving_yards": ["median", "mean"],
    "tracking_observations": ["median"],
    "maximum_sampling_gap_seconds": ["median", "max"],
    "gaps_over_0_15_seconds": ["sum"],
    "rows_with_missing_movement_values": ["sum"]
})

player_season_profile = (
    game_post
    .groupby(["draft_year", "season", "nfl_id"])
    .agg(game_aggregation)
)

# Flatten column names.
player_season_profile.columns = [
    f"game_{feature}_{stat}"
    for feature, stat in player_season_profile.columns
]

player_season_profile = player_season_profile.reset_index()

# -------------------------------------------------------
# 8. ADD EXPOSURE AND COVERAGE VARIABLES
# -------------------------------------------------------

exposure = (
    game_post
    .groupby(["draft_year", "season", "nfl_id"])
    .agg(
        receiver_plays=("play_id", "size"),
        games=("game_id", "nunique"),
        route_types_observed=("route_ran", "nunique"),
        plays_with_receiving_yards=("receiving_yards", "count"),
        plays_with_separation=("separation_at_pass_forward", "count")
    )
    .reset_index()
)

player_season_profile = player_season_profile.merge(
    exposure,
    on=["draft_year", "season", "nfl_id"],
    how="left",
    validate="one_to_one"
)

player_season_profile["receiving_yards_coverage_pct"] = (
    player_season_profile["plays_with_receiving_yards"]
    / player_season_profile["receiver_plays"] * 100
)

player_season_profile["separation_coverage_pct"] = (
    player_season_profile["plays_with_separation"]
    / player_season_profile["receiver_plays"] * 100
)

# -------------------------------------------------------
# 9. VALIDATE
# -------------------------------------------------------

assert not player_season_profile.duplicated(
    ["draft_year", "season", "nfl_id"]
).any()

print("PLAYER-SEASON MOVEMENT PROFILE")
print("Rows:", len(player_season_profile))
print("Unique players:", player_season_profile["nfl_id"].nunique())
print("Draft cohorts:",
      sorted(player_season_profile["draft_year"].unique()))

print("\nSEASON DISTRIBUTION")
display(
    player_season_profile.groupby("season")
    .agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique"),
        median_receiver_plays=("receiver_plays", "median")
    )
    .reset_index()
)

print("\nOUTCOME COVERAGE")
display(
    player_season_profile[
        [
            "season",
            "nfl_id",
            "receiver_plays",
            "plays_with_receiving_yards",
            "receiving_yards_coverage_pct",
            "plays_with_separation",
            "separation_coverage_pct"
        ]
    ].head(10)
)

# -------------------------------------------------------
# 10. SAVE PLAYER-SEASON DATASET
# -------------------------------------------------------

player_season_profile.to_parquet(
    FEATURES / "wr_game_player_season_profiles.parquet",
    index=False
)

player_season_profile.to_csv(
    FEATURES / "wr_game_player_season_profiles.csv",
    index=False
)

print("\nSAVED:")
print(FEATURES / "wr_game_player_season_profiles.parquet")
print(FEATURES / "wr_game_player_season_profiles.csv")


PLAYER-SEASON MOVEMENT PROFILE
Rows: 146
Unique players: 79
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

SEASON DISTRIBUTION


,season,player_seasons,unique_players,median_receiver_plays
0,2023,29,29,33.0
1,2024,52,52,34.0
2,2025,65,65,33.0



OUTCOME COVERAGE


,season,nfl_id,receiver_plays,plays_with_receiving_yards,receiving_yards_coverage_pct,plays_with_separation,separation_coverage_pct
0,2023,55884,90,63,70.000000,90,100.0
1,2023,55885,66,37,56.060606,66,100.0
2,2023,55887,103,65,63.106796,103,100.0
3,2023,55904,85,43,50.588235,85,100.0
4,2023,55915,82,54,65.853659,82,100.0
5,2023,55920,97,74,76.288660,97,100.0
6,2023,55928,33,22,66.666667,33,100.0
7,2023,55934,74,46,62.162162,74,100.0
8,2023,55938,40,23,57.500000,40,100.0
9,2023,55939,44,21,47.727273,44,100.0



SAVED:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_game_player_season_profiles.parquet
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_game_player_season_profiles.csv


In [16]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"

# Reload source game movement data
game = pd.read_parquet(
    FEATURES / "wr_game_movement_features.parquet"
)

# Recover the dominant position for each player-season
position_lookup = (
    game.groupby(["season", "nfl_id"])["position"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .reset_index(name="position")
)

# Remove any existing position column before restoring it
integrated = integrated.drop(
    columns=["position"],
    errors="ignore"
)

integrated = integrated.merge(
    position_lookup,
    on=["season", "nfl_id"],
    how="left",
    validate="one_to_one"
)

# Validate
print("INTEGRATED DATASET:", integrated.shape)
print("\nPOSITION DISTRIBUTION")
print(integrated["position"].value_counts(dropna=False))

print("\nPOSITION BY SEASON")
print(pd.crosstab(integrated["season"], integrated["position"]))

print("\nMISSING POSITIONS:", integrated["position"].isna().sum())

assert integrated["position"].notna().all()
assert integrated.duplicated(["draft_year", "season", "nfl_id"]).sum() == 0

# Save corrected integrated dataset
integrated.to_parquet(
    FEATURES / "wr_combine_game_integrated.parquet",
    index=False
)

integrated.to_csv(
    FEATURES / "wr_combine_game_integrated.csv",
    index=False
)

print("\nSAVED:")
print(FEATURES / "wr_combine_game_integrated.parquet")
print(FEATURES / "wr_combine_game_integrated.csv")

INTEGRATED DATASET: (146, 70)

POSITION DISTRIBUTION
position
WR         111
SLOT_WR     35
Name: count, dtype: int64

POSITION BY SEASON
position  SLOT_WR  WR
season               
2023            5  24
2024           17  35
2025           13  52

MISSING POSITIONS: 0

SAVED:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_game_integrated.parquet
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_game_integrated.csv


In [17]:
print("=" * 65)
print("WR-MOVEMENTIQ: INTEGRATED DATA QUALITY AUDIT")
print("=" * 65)

print("\n1. SAMPLE SIZE")
print("Player-seasons:", len(integrated))
print("Unique players:", integrated["nfl_id"].nunique())
print("Draft cohorts:", sorted(integrated["draft_year"].unique()))
print("Seasons:", sorted(integrated["season"].unique()))

print("\n2. POST-DRAFT VALIDATION")
print("Invalid season records:",
      (integrated["season"] < integrated["draft_year"]).sum())

print("\n3. EXPOSURE DISTRIBUTION")
print(integrated["receiver_plays"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
).round(2))

print("\n4. EXPOSURE SENSITIVITY")
for threshold in [1, 5, 10, 20, 30, 50]:
    subset = integrated[
        integrated["receiver_plays"] >= threshold
    ]
    print(
        f"Minimum {threshold:2} plays | "
        f"Rows: {len(subset):3} | "
        f"Players: {subset['nfl_id'].nunique():3}"
    )

print("\n5. OUTCOME COVERAGE")
coverage_cols = [
    c for c in integrated.columns
    if "coverage_pct" in c
]

print(integrated[coverage_cols].describe().round(2))

print("\n6. POSITION BY DRAFT COHORT")
print(pd.crosstab(
    integrated["draft_year"],
    integrated["position"]
))

print("\n7. DUPLICATE PLAYER-SEASONS")
print(
    integrated.duplicated(
        ["draft_year", "season", "nfl_id"]
    ).sum()
)

WR-MOVEMENTIQ: INTEGRATED DATA QUALITY AUDIT

1. SAMPLE SIZE
Player-seasons: 146
Unique players: 79
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]
Seasons: [np.int64(2023), np.int64(2024), np.int64(2025)]

2. POST-DRAFT VALIDATION
Invalid season records: 0

3. EXPOSURE DISTRIBUTION
count    146.00
mean      42.41
std       36.04
min        1.00
10%        4.00
25%       13.00
50%       33.00
75%       66.75
90%       90.00
max      156.00
Name: receiver_plays, dtype: float64

4. EXPOSURE SENSITIVITY
Minimum  1 plays | Rows: 146 | Players:  79
Minimum  5 plays | Rows: 129 | Players:  70
Minimum 10 plays | Rows: 116 | Players:  66
Minimum 20 plays | Rows:  93 | Players:  54
Minimum 30 plays | Rows:  80 | Players:  45
Minimum 50 plays | Rows:  54 | Players:  31

5. OUTCOME COVERAGE
       receiving_yards_coverage_pct  separation_coverage_pct
count                        146.00                    146.0
mean                          60.74                    100.0
std       

In [18]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"

integrated = pd.read_parquet(
    FEATURES / "wr_combine_game_integrated.parquet"
)

print("=" * 75)
print("WR-MOVEMENTIQ: MODELLING VARIABLE READINESS")
print("=" * 75)

# 1. Inspect all available variables
print("\n1. COMPLETE COLUMN INVENTORY")
for col in integrated.columns:
    print(col)

# 2. Identify candidate in-game outcomes
candidate_terms = [
    "mean_speed",
    "peak_speed",
    "mean_acceleration",
    "path_efficiency",
    "direction_change",
    "separation",
    "receiving_yards"
]

outcome_columns = [
    col for col in integrated.columns
    if col.startswith("game_")
    and any(term in col for term in candidate_terms)
]

print("\n2. CANDIDATE GAME OUTCOMES")
for col in outcome_columns:
    print(col)

# 3. Numeric distribution and missingness
print("\n3. OUTCOME DISTRIBUTIONS")

for col in outcome_columns:
    values = pd.to_numeric(integrated[col], errors="coerce")

    print(f"\n{col}")
    print("Nonmissing:", values.notna().sum())
    print("Missing:", values.isna().sum())
    print("Missing %:", round(values.isna().mean() * 100, 2))
    print(values.describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).round(4))

# 4. Combine predictor inventory
print("\n4. COMBINE PROFILE VARIABLES")

combine_columns = [
    col for col in integrated.columns
    if col not in [
        "draft_year", "nfl_id", "season", "position"
    ]
    and not col.startswith("game_")
]

for col in combine_columns:
    print(col)

# 5. Check predictor missingness
print("\n5. COMBINE PREDICTOR MISSINGNESS")

combine_numeric = integrated[combine_columns].select_dtypes(
    include=np.number
)

missing_summary = pd.DataFrame({
    "missing_count": combine_numeric.isna().sum(),
    "missing_pct": combine_numeric.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

print(missing_summary.round(2).to_string())

# 6. Exposure-restricted sample
primary_sample = integrated[
    integrated["receiver_plays"] >= 10
].copy()

sensitivity_sample = integrated[
    integrated["receiver_plays"] >= 20
].copy()

print("\n6. MODELLING SAMPLE SIZES")
for name, sample in [
    ("All observations", integrated),
    ("Primary: >=10 plays", primary_sample),
    ("Sensitivity: >=20 plays", sensitivity_sample)
]:
    print(
        name,
        "| Rows:", len(sample),
        "| Players:", sample["nfl_id"].nunique()
    )

# 7. Repeated observations per player
player_seasons = (
    integrated.groupby("nfl_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\n7. NUMBER OF SEASONS PER PLAYER")
print(player_seasons)

# 8. Save audit
audit_path = ROOT / "reports" / "data_audit"
audit_path.mkdir(parents=True, exist_ok=True)

missing_summary.to_csv(
    audit_path / "model_predictor_missingness.csv"
)

print("\nAUDIT SAVED:", audit_path)


WR-MOVEMENTIQ: MODELLING VARIABLE READINESS

1. COMPLETE COLUMN INVENTORY
draft_year
nfl_id
primary_drills_observed
primary_attempts_observed
combine_mean_speed_median_across_drills
combine_mean_speed_iqr_across_drills
combine_peak_speed_median_across_drills
combine_peak_speed_iqr_across_drills
combine_p95_speed_median_across_drills
combine_p95_speed_iqr_across_drills
combine_mean_acceleration_median_across_drills
combine_mean_acceleration_iqr_across_drills
combine_peak_acceleration_median_across_drills
combine_peak_acceleration_iqr_across_drills
combine_coordinate_path_length_median_across_drills
combine_coordinate_path_length_iqr_across_drills
combine_net_displacement_median_across_drills
combine_net_displacement_iqr_across_drills
combine_path_efficiency_median_across_drills
combine_path_efficiency_iqr_across_drills
combine_coordinate_total_turn_median_across_drills
combine_coordinate_total_turn_iqr_across_drills
combine_coordinate_mean_turn_median_across_drills
combine_coordinate_me

In [19]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
RAW = ROOT / "data" / "raw" / "nfl-big-data-bowl-2027"
FEATURES = ROOT / "data" / "features"
AUDIT = ROOT / "reports" / "data_audit"

integrated = pd.read_parquet(
    FEATURES / "wr_combine_game_integrated.parquet"
)

combine = pd.read_csv(RAW / "combine_results.csv")

# Restrict source to wide receivers
combine_wr = combine[
    combine["combine_position"].astype(str).str.upper().eq("WR")
].copy()

# Traditional Combine variables
traditional_candidates = [
    "ten_yd_split",
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps",
    "ngs_athleticism_score",
    "ngs_college_production_score",
    "ngs_final_score"
]

traditional_features = [
    c for c in traditional_candidates if c in combine_wr.columns
]

# Normalize numeric-like source columns
for col in traditional_features:
    combine_wr[col] = pd.to_numeric(
        combine_wr[col], errors="coerce"
    )

# Verify key uniqueness before merging
duplicate_keys = combine_wr.duplicated(
    ["draft_year", "nfl_id"], keep=False
)

print("WR source rows:", len(combine_wr))
print("Unique WR players:", combine_wr["nfl_id"].nunique())
print("Duplicate player-draft keys:", duplicate_keys.sum())

if duplicate_keys.any():
    print("\nDUPLICATE RECORDS")
    print(
        combine_wr.loc[
            duplicate_keys,
            ["draft_year", "nfl_id", "combine_position"]
            + traditional_features
        ].sort_values(["draft_year", "nfl_id"]).to_string(index=False)
    )
    raise ValueError(
        "Resolve duplicate Combine player-draft keys before merging."
    )

# Retain only identifiers and traditional features
traditional = combine_wr[
    ["draft_year", "nfl_id"] + traditional_features
].copy()

# Avoid duplicate fields if this cell is rerun
integrated = integrated.drop(
    columns=traditional_features,
    errors="ignore"
)

# Merge onto player-season observations
integrated = integrated.merge(
    traditional,
    on=["draft_year", "nfl_id"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nMERGE STATUS")
print(integrated["_merge"].value_counts())

integrated = integrated.drop(columns="_merge")

# Coverage audit
coverage = pd.DataFrame({
    "available": integrated[traditional_features].notna().sum(),
    "missing": integrated[traditional_features].isna().sum(),
    "coverage_pct": (
        integrated[traditional_features].notna().mean() * 100
    ).round(2)
}).sort_values("coverage_pct", ascending=False)

print("\nTRADITIONAL COMBINE COVERAGE")
print(coverage)

print("\nTRADITIONAL COMBINE DISTRIBUTIONS")
print(
    integrated[traditional_features]
    .describe()
    .round(3)
    .T
)

# Save integrated version with traditional metrics
integrated.to_parquet(
    FEATURES / "wr_combine_game_integrated_v2.parquet",
    index=False
)

integrated.to_csv(
    FEATURES / "wr_combine_game_integrated_v2.csv",
    index=False
)

coverage.to_csv(
    AUDIT / "traditional_combine_coverage.csv"
)

print("\nSAVED:")
print(FEATURES / "wr_combine_game_integrated_v2.parquet")
print(FEATURES / "wr_combine_game_integrated_v2.csv")


WR source rows: 107
Unique WR players: 107
Duplicate player-draft keys: 0

MERGE STATUS
_merge
both          146
left_only       0
right_only      0
Name: count, dtype: int64

TRADITIONAL COMBINE COVERAGE
                              available  missing  coverage_pct
ngs_final_score                     146        0        100.00
ngs_college_production_score        146        0        100.00
ngs_athleticism_score               146        0        100.00
ten_yd_split                        131       15         89.73
forty                               131       15         89.73
broad_jump                          130       16         89.04
vertical                            128       18         87.67
short_shuttle                        58       88         39.73
three_cone                           48       98         32.88
bench_reps                           38      108         26.03

TRADITIONAL COMBINE DISTRIBUTIONS
                              count     mean     std     min      2

In [20]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
AUDIT = ROOT / "reports" / "data_audit"
AUDIT.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(
    FEATURES / "wr_combine_game_integrated_v2.parquet"
)

# Primary research sample
sample = df[df["receiver_plays"] >= 10].copy()

# Candidate outcomes
OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_total_direction_change_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

# Model A: traditional Combine
TRADITIONAL = [
    "forty",
    "vertical",
    "broad_jump",
    "ngs_athleticism_score",
    "ngs_college_production_score",
    "ngs_final_score"
]

# Model B: tracking-derived Combine mechanics
TRACKING = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
    "combine_coordinate_total_turn_median_across_drills",
    "combine_coordinate_p95_turn_median_across_drills"
]

# Confirm columns
for col in OUTCOMES + TRADITIONAL + TRACKING:
    if col not in sample.columns:
        raise KeyError(f"Missing expected variable: {col}")

print("=" * 70)
print("PRIMARY MODELLING SAMPLE")
print("=" * 70)
print("Rows:", len(sample))
print("Unique players:", sample["nfl_id"].nunique())
print("Draft cohorts:", sorted(sample["draft_year"].unique()))

print("\nPLAYER-SEASON OBSERVATIONS PER PLAYER")
print(sample.groupby("nfl_id").size().value_counts().sort_index())

# 1. Missingness
candidate_columns = OUTCOMES + TRADITIONAL + TRACKING

missingness = pd.DataFrame({
    "missing": sample[candidate_columns].isna().sum(),
    "missing_pct": sample[candidate_columns].isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

print("\nMISSINGNESS")
print(missingness.round(2))

# 2. Predictor correlations
traditional_corr = sample[TRADITIONAL].corr(method="spearman")
tracking_corr = sample[TRACKING].corr(method="spearman")

print("\nMODEL A: TRADITIONAL PREDICTOR CORRELATIONS")
print(traditional_corr.round(2))

print("\nMODEL B: TRACKING PREDICTOR CORRELATIONS")
print(tracking_corr.round(2))

traditional_corr.to_csv(
    AUDIT / "traditional_predictor_spearman_correlations.csv"
)

tracking_corr.to_csv(
    AUDIT / "tracking_predictor_spearman_correlations.csv"
)

# 3. Strong predictor pairs
def strong_pairs(corr, threshold=0.80):
    pairs = []
    cols = corr.columns

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            value = corr.iloc[i, j]
            if pd.notna(value) and abs(value) >= threshold:
                pairs.append({
                    "feature_1": cols[i],
                    "feature_2": cols[j],
                    "spearman_rho": value
                })

    return pd.DataFrame(pairs)

print("\nSTRONGLY CORRELATED TRADITIONAL PAIRS")
print(strong_pairs(traditional_corr).round(3).to_string(index=False))

print("\nSTRONGLY CORRELATED TRACKING PAIRS")
print(strong_pairs(tracking_corr).round(3).to_string(index=False))

# 4. Outcome correlations with candidate predictors
all_predictors = TRADITIONAL + TRACKING

outcome_correlations = []

for outcome in OUTCOMES:
    for predictor in all_predictors:
        valid = sample[[outcome, predictor]].dropna()

        rho = (
            valid[outcome].corr(valid[predictor], method="spearman")
            if len(valid) >= 5 else np.nan
        )

        outcome_correlations.append({
            "outcome": outcome,
            "predictor": predictor,
            "spearman_rho": rho,
            "n_player_seasons": len(valid),
            "n_players": valid.index.map(
                sample["nfl_id"]
            ).nunique() if len(valid) else 0
        })

outcome_corr_df = pd.DataFrame(outcome_correlations)

print("\nOUTCOME-PREDICTOR ASSOCIATIONS")
print(
    outcome_corr_df.sort_values(
        ["outcome", "spearman_rho"],
        ascending=[True, False]
    ).round(3).to_string(index=False)
)

outcome_corr_df.to_csv(
    AUDIT / "outcome_predictor_spearman_correlations.csv",
    index=False
)

# 5. Feature variation and complete-case readiness
print("\nPREDICTOR VARIATION")
for col in all_predictors:
    print(
        col,
        "| Unique values:", sample[col].nunique(dropna=True),
        "| Missing:", sample[col].isna().sum()
    )

# 6. Save the primary modelling sample
sample.to_parquet(
    FEATURES / "wr_movementiq_primary_sample.parquet",
    index=False
)

sample.to_csv(
    FEATURES / "wr_movementiq_primary_sample.csv",
    index=False
)

print("\nSAVED PRIMARY SAMPLE")
print(FEATURES / "wr_movementiq_primary_sample.parquet")
print(FEATURES / "wr_movementiq_primary_sample.csv")


PRIMARY MODELLING SAMPLE
Rows: 116
Unique players: 66
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

PLAYER-SEASON OBSERVATIONS PER PLAYER
1    30
2    22
3    14
Name: count, dtype: int64

MISSINGNESS
                                                    missing  missing_pct
vertical                                                 14        12.07
forty                                                    12        10.34
broad_jump                                               12        10.34
game_mean_speed_raw_median                                0         0.00
game_peak_speed_raw_median                                0         0.00
game_mean_acceleration_raw_median                         0         0.00
game_separation_at_pass_forward_median                    0         0.00
game_path_efficiency_median                               0         0.00
game_total_direction_change_raw_median                    0         0.00
ngs_athleticism_score                             

In [21]:

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
RESULTS = ROOT / "reports" / "model_results"
RESULTS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(
    FEATURES / "wr_movementiq_primary_sample.parquet"
).copy()

# ---------------------------------------------------------
# 1. Define modelling specifications
# ---------------------------------------------------------

MODEL_A = [
    "forty",
    "vertical",
    "broad_jump"
]

MODEL_B = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
    "combine_coordinate_total_turn_median_across_drills",
    "combine_coordinate_p95_turn_median_across_drills"
]

MODEL_C = MODEL_A + MODEL_B

MODELS = {
    "A_Traditional": MODEL_A,
    "B_Tracking": MODEL_B,
    "C_Combined": MODEL_C
}

# Primary outcome first; remaining outcomes are secondary
OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_total_direction_change_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

# Validate
required = MODEL_C + OUTCOMES + ["nfl_id", "receiver_plays"]
missing_cols = [c for c in required if c not in df.columns]

if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

assert df["nfl_id"].nunique() == 66
assert len(df) == 116

# ---------------------------------------------------------
# 2. Player-balanced weights
# ---------------------------------------------------------

seasons_per_player = df.groupby("nfl_id")["nfl_id"].transform("size")
df["player_weight"] = 1.0 / seasons_per_player

# ---------------------------------------------------------
# 3. Fixed grouped folds
# ---------------------------------------------------------

cv = GroupKFold(n_splits=5)

folds = list(
    cv.split(
        df,
        groups=df["nfl_id"]
    )
)

print("FOLDS")
for fold_number, (train_idx, test_idx) in enumerate(folds, 1):
    train_players = df.iloc[train_idx]["nfl_id"].nunique()
    test_players = df.iloc[test_idx]["nfl_id"].nunique()

    overlap = set(df.iloc[train_idx]["nfl_id"]) & set(
        df.iloc[test_idx]["nfl_id"]
    )

    print(
        f"Fold {fold_number}: "
        f"train rows={len(train_idx)}, "
        f"train players={train_players}, "
        f"test rows={len(test_idx)}, "
        f"test players={test_players}, "
        f"player overlap={len(overlap)}"
    )

    assert len(overlap) == 0

# ---------------------------------------------------------
# 4. Fit and evaluate
# ---------------------------------------------------------

all_predictions = []
summary_rows = []

for outcome in OUTCOMES:

    y = pd.to_numeric(df[outcome], errors="coerce")

    if y.isna().any():
        raise ValueError(f"Missing target values in {outcome}")

    for model_name, predictors in MODELS.items():

        oof = np.full(len(df), np.nan)

        for fold_number, (train_idx, test_idx) in enumerate(folds, 1):

            X_train = df.iloc[train_idx][predictors]
            X_test = df.iloc[test_idx][predictors]

            y_train = y.iloc[train_idx]
            w_train = df.iloc[train_idx]["player_weight"]

            # All preprocessing is fitted within the training fold
            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=1.0))
            ])

            pipeline.fit(
                X_train,
                y_train,
                model__sample_weight=w_train
            )

            oof[test_idx] = pipeline.predict(X_test)

        assert not np.isnan(oof).any()

        weights = df["player_weight"].to_numpy()
        actual = y.to_numpy()

        mae = mean_absolute_error(
            actual, oof, sample_weight=weights
        )

        rmse = np.sqrt(
            mean_squared_error(
                actual, oof, sample_weight=weights
            )
        )

        r2 = r2_score(
            actual, oof, sample_weight=weights
        )

        rho, p_value = spearmanr(actual, oof)

        summary_rows.append({
            "outcome": outcome,
            "model": model_name,
            "n_player_seasons": len(df),
            "n_players": df["nfl_id"].nunique(),
            "mae_player_weighted": mae,
            "rmse_player_weighted": rmse,
            "r2_player_weighted": r2,
            "oof_spearman": rho,
            "spearman_p_value": p_value,
            "ridge_alpha": 1.0
        })

        for i in range(len(df)):
            all_predictions.append({
                "nfl_id": df.iloc[i]["nfl_id"],
                "draft_year": df.iloc[i]["draft_year"],
                "season": df.iloc[i]["season"],
                "outcome": outcome,
                "model": model_name,
                "actual": actual[i],
                "predicted": oof[i],
                "player_weight": weights[i]
            })

# ---------------------------------------------------------
# 5. Results
# ---------------------------------------------------------

results = pd.DataFrame(summary_rows)

print("\n" + "=" * 75)
print("WR-MOVEMENTIQ: GROUPED RIDGE VALIDATION RESULTS")
print("=" * 75)

print(
    results.sort_values(
        ["outcome", "mae_player_weighted"]
    ).round(4).to_string(index=False)
)

predictions = pd.DataFrame(all_predictions)

# Save
results.to_csv(
    RESULTS / "grouped_ridge_model_comparison.csv",
    index=False
)

predictions.to_parquet(
    RESULTS / "grouped_ridge_oof_predictions.parquet",
    index=False
)

predictions.to_csv(
    RESULTS / "grouped_ridge_oof_predictions.csv",
    index=False
)

print("\nSAVED RESULTS:")
print(RESULTS / "grouped_ridge_model_comparison.csv")
print(RESULTS / "grouped_ridge_oof_predictions.parquet")
print(RESULTS / "grouped_ridge_oof_predictions.csv")


FOLDS
Fold 1: train rows=92, train players=52, test rows=24, test players=14, player overlap=0
Fold 2: train rows=93, train players=53, test rows=23, test players=13, player overlap=0
Fold 3: train rows=93, train players=53, test rows=23, test players=13, player overlap=0
Fold 4: train rows=93, train players=53, test rows=23, test players=13, player overlap=0
Fold 5: train rows=93, train players=53, test rows=23, test players=13, player overlap=0

WR-MOVEMENTIQ: GROUPED RIDGE VALIDATION RESULTS
                               outcome         model  n_player_seasons  n_players  mae_player_weighted  rmse_player_weighted  r2_player_weighted  oof_spearman  spearman_p_value  ridge_alpha
     game_mean_acceleration_raw_median A_Traditional               116         66               0.1640                0.2043             -0.0182        0.0474            0.6134          1.0
     game_mean_acceleration_raw_median    B_Tracking               116         66               0.1666                0.

In [22]:

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
RESULTS = ROOT / "reports" / "model_results"
RESULTS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(
    FEATURES / "wr_movementiq_primary_sample.parquet"
).copy().reset_index(drop=True)

MODEL_A = ["forty", "vertical", "broad_jump"]

MODEL_B = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
    "combine_coordinate_total_turn_median_across_drills",
    "combine_coordinate_p95_turn_median_across_drills"
]

MODELS = {
    "A_Traditional": MODEL_A,
    "B_Tracking": MODEL_B,
    "C_Combined": MODEL_A + MODEL_B
}

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_total_direction_change_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

# Equal total training weight per player
player_counts = df.groupby("nfl_id")["nfl_id"].transform("size")
df["player_weight"] = 1.0 / player_counts

cv = GroupKFold(n_splits=5)
folds = list(cv.split(df, groups=df["nfl_id"]))

records = []

for outcome in OUTCOMES:

    y = pd.to_numeric(df[outcome], errors="coerce")

    if y.isna().any():
        raise ValueError(f"Missing outcome values: {outcome}")

    for fold_no, (train_idx, test_idx) in enumerate(folds, 1):

        train = df.iloc[train_idx]
        test = df.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Fold-specific weighted training mean baseline
        baseline = np.average(
            y_train,
            weights=train["player_weight"]
        )

        for model_name, predictors in MODELS.items():

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=1.0))
            ])

            pipeline.fit(
                train[predictors],
                y_train,
                ridge__sample_weight=train["player_weight"]
            )

            predictions = pipeline.predict(test[predictors])

            for position, idx in enumerate(test_idx):
                records.append({
                    "row_index": idx,
                    "nfl_id": df.loc[idx, "nfl_id"],
                    "draft_year": df.loc[idx, "draft_year"],
                    "season": df.loc[idx, "season"],
                    "fold": fold_no,
                    "outcome": outcome,
                    "model": model_name,
                    "actual": y.loc[idx],
                    "predicted": predictions[position],
                    "baseline_predicted": baseline,
                    "player_weight": df.loc[idx, "player_weight"]
                })

        # Store baseline as its own model
        for position, idx in enumerate(test_idx):
            records.append({
                "row_index": idx,
                "nfl_id": df.loc[idx, "nfl_id"],
                "draft_year": df.loc[idx, "draft_year"],
                "season": df.loc[idx, "season"],
                "fold": fold_no,
                "outcome": outcome,
                "model": "Baseline",
                "actual": y.loc[idx],
                "predicted": baseline,
                "baseline_predicted": baseline,
                "player_weight": df.loc[idx, "player_weight"]
            })

oof = pd.DataFrame(records)

# ---------------------------------------------------------
# A. Player-season metrics
# ---------------------------------------------------------

season_results = []

for (outcome, model), group in oof.groupby(["outcome", "model"]):

    actual = group["actual"].to_numpy()
    predicted = group["predicted"].to_numpy()
    weights = group["player_weight"].to_numpy()

    season_results.append({
        "outcome": outcome,
        "model": model,
        "n_player_seasons": len(group),
        "n_players": group["nfl_id"].nunique(),
        "weighted_mae": mean_absolute_error(
            actual, predicted, sample_weight=weights
        ),
        "weighted_rmse": np.sqrt(
            mean_squared_error(
                actual, predicted, sample_weight=weights
            )
        ),
        "weighted_r2": r2_score(
            actual, predicted, sample_weight=weights
        )
    })

season_results = pd.DataFrame(season_results)

# ---------------------------------------------------------
# B. Aggregate actual and predicted values per player
# ---------------------------------------------------------

player_oof = (
    oof.groupby(
        ["outcome", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

player_results = []

for (outcome, model), group in player_oof.groupby(["outcome", "model"]):

    actual = group["actual"].to_numpy()
    predicted = group["predicted"].to_numpy()

    rho = spearmanr(actual, predicted).statistic

    player_results.append({
        "outcome": outcome,
        "model": model,
        "n_players": len(group),
        "player_mae": mean_absolute_error(actual, predicted),
        "player_rmse": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "player_r2": r2_score(actual, predicted),
        "player_level_spearman": rho
    })

player_results = pd.DataFrame(player_results)

# ---------------------------------------------------------
# C. Compare each model with the fold-specific baseline
# ---------------------------------------------------------

comparison = player_results.pivot(
    index="outcome",
    columns="model",
    values=["player_mae", "player_rmse", "player_r2",
            "player_level_spearman"]
)

print("=" * 80)
print("PLAYER-LEVEL OUT-OF-FOLD VALIDATION")
print("=" * 80)
print(player_results.round(4).to_string(index=False))

print("\n" + "=" * 80)
print("PLAYER-SEASON WEIGHTED VALIDATION")
print("=" * 80)
print(season_results.round(4).to_string(index=False))

# MAE improvement relative to baseline
mae_table = player_results.pivot(
    index="outcome",
    columns="model",
    values="player_mae"
)

print("\nPLAYER-LEVEL MAE IMPROVEMENT VS BASELINE")
for model in ["A_Traditional", "B_Tracking", "C_Combined"]:
    improvement = (
        (mae_table["Baseline"] - mae_table[model])
        / mae_table["Baseline"] * 100
    )
    print(f"\n{model}")
    print(improvement.round(2).to_string())

# Save all outputs
oof.to_parquet(
    RESULTS / "grouped_ridge_oof_predictions_v2.parquet",
    index=False
)

oof.to_csv(
    RESULTS / "grouped_ridge_oof_predictions_v2.csv",
    index=False
)

player_oof.to_csv(
    RESULTS / "grouped_ridge_player_level_predictions.csv",
    index=False
)

player_results.to_csv(
    RESULTS / "grouped_ridge_player_level_metrics.csv",
    index=False
)

season_results.to_csv(
    RESULTS / "grouped_ridge_player_season_metrics_v2.csv",
    index=False
)

print("\nCORRECTED VALIDATION OUTPUTS SAVED.")


PLAYER-LEVEL OUT-OF-FOLD VALIDATION
                               outcome         model  n_players  player_mae  player_rmse  player_r2  player_level_spearman
     game_mean_acceleration_raw_median A_Traditional         66      0.1464       0.1823    -0.0230                 0.0905
     game_mean_acceleration_raw_median    B_Tracking         66      0.1482       0.1877    -0.0839                 0.0877
     game_mean_acceleration_raw_median      Baseline         66      0.1439       0.1805    -0.0033                -0.0394
     game_mean_acceleration_raw_median    C_Combined         66      0.1541       0.1897    -0.1074                 0.0925
            game_mean_speed_raw_median A_Traditional         66      0.3927       0.5228    -0.0609                 0.1087
            game_mean_speed_raw_median    B_Tracking         66      0.3908       0.5519    -0.1825                -0.0902
            game_mean_speed_raw_median      Baseline         66      0.3725       0.5245    -0.0680    

In [25]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"

# Start with the existing >=10-play sample
df20 = pd.read_parquet(
    FEATURES / "wr_movementiq_primary_sample.parquet"
).copy()

# Apply stricter exposure threshold
df20 = df20[df20["receiver_plays"] >= 20].copy()

print("SENSITIVITY SAMPLE: 20+ RECEIVER PLAYS")
print("Player-seasons:", len(df20))
print("Unique players:", df20["nfl_id"].nunique())
print("Draft cohorts:", sorted(df20["draft_year"].unique()))
print("\nSeasons per player:")
print(df20.groupby("nfl_id")["season"].nunique().value_counts().sort_index())

# Save as a distinct dataset
df20.to_parquet(
    FEATURES / "wr_movementiq_sensitivity_20plus.parquet",
    index=False
)

df20.to_csv(
    FEATURES / "wr_movementiq_sensitivity_20plus.csv",
    index=False
)

print("\nSaved 20+ play sensitivity dataset.")


SENSITIVITY SAMPLE: 20+ RECEIVER PLAYS
Player-seasons: 93
Unique players: 54
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

Seasons per player:
season
1    26
2    17
3    11
Name: count, dtype: int64

Saved 20+ play sensitivity dataset.


In [26]:
df = pd.read_parquet(
    FEATURES / "wr_movementiq_sensitivity_20plus.parquet"
).copy().reset_index(drop=True)

In [27]:
"grouped_ridge_oof_predictions_20plus.parquet"
"grouped_ridge_oof_predictions_20plus.csv"
"grouped_ridge_player_level_predictions_20plus.csv"
"grouped_ridge_player_level_metrics_20plus.csv"
"grouped_ridge_player_season_metrics_20plus.csv"

'grouped_ridge_player_season_metrics_20plus.csv'

In [28]:
assert len(df) == 93
assert df["nfl_id"].nunique() == 54
assert df.groupby("nfl_id")["season"].nunique().max() <= 3

print("20+ sample verified.")

20+ sample verified.


In [30]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
RESULTS = ROOT / "reports" / "model_results"

RESULTS.mkdir(parents=True, exist_ok=True)

DATA = FEATURES / "wr_movementiq_sensitivity_20plus.parquet"

assert DATA.exists(), f"Missing input dataset: {DATA}"

df = pd.read_parquet(DATA).reset_index(drop=True)

print("20+ DATASET")
print("Rows:", len(df))
print("Players:", df["nfl_id"].nunique())
print("Columns:", len(df.columns))


20+ DATASET
Rows: 93
Players: 54
Columns: 80


In [31]:

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

MODEL_A = ["forty", "vertical", "broad_jump"]

MODEL_B = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
    "combine_coordinate_total_turn_median_across_drills",
    "combine_coordinate_p95_turn_median_across_drills"
]

MODELS = {
    "A_Traditional": MODEL_A,
    "B_Tracking": MODEL_B,
    "C_Combined": MODEL_A + MODEL_B
}

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_total_direction_change_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

assert len(df) == 93
assert df["nfl_id"].nunique() == 54

# Equal total weight per player
df["player_weight"] = (
    1 / df.groupby("nfl_id")["nfl_id"].transform("size")
)

cv = GroupKFold(n_splits=5)
folds = list(cv.split(df, groups=df["nfl_id"]))

records = []

for outcome in OUTCOMES:

    y = pd.to_numeric(df[outcome], errors="coerce")

    if y.isna().any():
        raise ValueError(f"Missing outcome values: {outcome}")

    for fold_no, (train_idx, test_idx) in enumerate(folds, 1):

        train = df.iloc[train_idx]
        test = df.iloc[test_idx]

        overlap = set(train["nfl_id"]) & set(test["nfl_id"])
        assert len(overlap) == 0

        baseline = np.average(
            y.iloc[train_idx],
            weights=train["player_weight"]
        )

        for model_name, predictors in MODELS.items():

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=1.0))
            ])

            pipeline.fit(
                train[predictors],
                y.iloc[train_idx],
                ridge__sample_weight=train["player_weight"]
            )

            predictions = pipeline.predict(test[predictors])

            for j, idx in enumerate(test_idx):
                records.append({
                    "row_index": idx,
                    "nfl_id": df.loc[idx, "nfl_id"],
                    "draft_year": df.loc[idx, "draft_year"],
                    "season": df.loc[idx, "season"],
                    "fold": fold_no,
                    "outcome": outcome,
                    "model": model_name,
                    "actual": y.loc[idx],
                    "predicted": predictions[j],
                    "baseline_predicted": baseline,
                    "player_weight": df.loc[idx, "player_weight"]
                })

        for idx in test_idx:
            records.append({
                "row_index": idx,
                "nfl_id": df.loc[idx, "nfl_id"],
                "draft_year": df.loc[idx, "draft_year"],
                "season": df.loc[idx, "season"],
                "fold": fold_no,
                "outcome": outcome,
                "model": "Baseline",
                "actual": y.loc[idx],
                "predicted": baseline,
                "baseline_predicted": baseline,
                "player_weight": df.loc[idx, "player_weight"]
            })

oof = pd.DataFrame(records)

oof.to_parquet(
    RESULTS / "grouped_ridge_oof_predictions_20plus.parquet",
    index=False
)

oof.to_csv(
    RESULTS / "grouped_ridge_oof_predictions_20plus.csv",
    index=False
)

print("20+ MODEL VALIDATION COMPLETE")
print("Prediction rows:", len(oof))
print("Unique players:", oof["nfl_id"].nunique())
print("Saved:", RESULTS / "grouped_ridge_oof_predictions_20plus.parquet")


20+ MODEL VALIDATION COMPLETE
Prediction rows: 2232
Unique players: 54
Saved: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/grouped_ridge_oof_predictions_20plus.parquet


In [32]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

player_oof = (
    oof.groupby(["outcome", "model", "nfl_id"], as_index=False)
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

results = []

for (outcome, model), g in player_oof.groupby(["outcome", "model"]):
    results.append({
        "outcome": outcome,
        "model": model,
        "n_players": len(g),
        "player_mae": mean_absolute_error(g.actual, g.predicted),
        "player_rmse": np.sqrt(mean_squared_error(g.actual, g.predicted)),
        "player_r2": r2_score(g.actual, g.predicted),
        "player_level_spearman": spearmanr(
            g.actual, g.predicted
        ).statistic
    })

player_metrics = pd.DataFrame(results)

player_metrics.to_csv(
    RESULTS / "grouped_ridge_player_level_metrics_20plus.csv",
    index=False
)

player_oof.to_csv(
    RESULTS / "grouped_ridge_player_level_predictions_20plus.csv",
    index=False
)

display(player_metrics.round(4))
print("Player-level metrics saved.")


,outcome,model,n_players,player_mae,player_rmse,player_r2,player_level_spearman
0,game_mean_acceleration_raw_median,A_Traditional,54,0.1346,0.1640,-0.0999,-0.0890
1,game_mean_acceleration_raw_median,B_Tracking,54,0.1396,0.1683,-0.1585,-0.0543
2,game_mean_acceleration_raw_median,Baseline,54,0.1297,0.1595,-0.0397,-0.2234
3,game_mean_acceleration_raw_median,C_Combined,54,0.1507,0.1818,-0.3514,-0.1455
4,game_mean_speed_raw_median,A_Traditional,54,0.3528,0.4432,-0.0490,0.0799
5,game_mean_speed_raw_median,B_Tracking,54,0.3588,0.4615,-0.1376,-0.1045
6,game_mean_speed_raw_median,Baseline,54,0.3440,0.4448,-0.0566,-0.3306
7,game_mean_speed_raw_median,C_Combined,54,0.3624,0.4415,-0.0412,0.1510
8,game_path_efficiency_median,A_Traditional,54,0.0076,0.0096,-0.1797,-0.3694
9,game_path_efficiency_median,B_Tracking,54,0.0077,0.0099,-0.2451,0.0472


Player-level metrics saved.


In [33]:

season_results = []

for (outcome, model), g in oof.groupby(["outcome", "model"]):

    y = g["actual"].to_numpy()
    pred = g["predicted"].to_numpy()
    weights = g["player_weight"].to_numpy()

    season_results.append({
        "outcome": outcome,
        "model": model,
        "n_player_seasons": len(g),
        "n_players": g["nfl_id"].nunique(),
        "weighted_mae": mean_absolute_error(
            y, pred, sample_weight=weights
        ),
        "weighted_rmse": np.sqrt(
            mean_squared_error(y, pred, sample_weight=weights)
        ),
        "weighted_r2": r2_score(
            y, pred, sample_weight=weights
        )
    })

season_metrics = pd.DataFrame(season_results)

season_metrics.to_csv(
    RESULTS / "grouped_ridge_player_season_metrics_20plus.csv",
    index=False
)

display(season_metrics.round(4))
print("Player-season metrics saved.")


,outcome,model,n_player_seasons,n_players,weighted_mae,weighted_rmse,weighted_r2
0,game_mean_acceleration_raw_median,A_Traditional,93,54,0.1457,0.1804,-0.0811
1,game_mean_acceleration_raw_median,B_Tracking,93,54,0.1495,0.1843,-0.1288
2,game_mean_acceleration_raw_median,Baseline,93,54,0.1418,0.1763,-0.0323
3,game_mean_acceleration_raw_median,C_Combined,93,54,0.1604,0.1967,-0.2855
4,game_mean_speed_raw_median,A_Traditional,93,54,0.3825,0.4755,-0.0423
5,game_mean_speed_raw_median,B_Tracking,93,54,0.3943,0.4927,-0.1187
6,game_mean_speed_raw_median,Baseline,93,54,0.3746,0.4770,-0.0488
7,game_mean_speed_raw_median,C_Combined,93,54,0.3896,0.4740,-0.0355
8,game_path_efficiency_median,A_Traditional,93,54,0.0081,0.0101,-0.1599
9,game_path_efficiency_median,B_Tracking,93,54,0.0084,0.0104,-0.2181


Player-season metrics saved.


In [34]:
for name in [
    "grouped_ridge_player_level_metrics_20plus.csv",
    "grouped_ridge_player_season_metrics_20plus.csv"
]:
    path = RESULTS / name
    print(name, "| Exists:", path.exists())

grouped_ridge_player_level_metrics_20plus.csv | Exists: True
grouped_ridge_player_season_metrics_20plus.csv | Exists: True


In [35]:

import pandas as pd

player = pd.read_csv(
    RESULTS / "grouped_ridge_player_level_metrics_20plus.csv"
)

season = pd.read_csv(
    RESULTS / "grouped_ridge_player_season_metrics_20plus.csv"
)

print("=" * 85)
print("20+ RECEIVER PLAYS: PLAYER-LEVEL METRICS")
print("=" * 85)
display(player.round(4))

print("\n" + "=" * 85)
print("20+ RECEIVER PLAYS: PLAYER-SEASON METRICS")
print("=" * 85)
display(season.round(4))


20+ RECEIVER PLAYS: PLAYER-LEVEL METRICS


,outcome,model,n_players,player_mae,player_rmse,player_r2,player_level_spearman
0,game_mean_acceleration_raw_median,A_Traditional,54,0.1346,0.1640,-0.0999,-0.0890
1,game_mean_acceleration_raw_median,B_Tracking,54,0.1396,0.1683,-0.1585,-0.0543
2,game_mean_acceleration_raw_median,Baseline,54,0.1297,0.1595,-0.0397,-0.2234
3,game_mean_acceleration_raw_median,C_Combined,54,0.1507,0.1818,-0.3514,-0.1455
4,game_mean_speed_raw_median,A_Traditional,54,0.3528,0.4432,-0.0490,0.0799
5,game_mean_speed_raw_median,B_Tracking,54,0.3588,0.4615,-0.1376,-0.1045
6,game_mean_speed_raw_median,Baseline,54,0.3440,0.4448,-0.0566,-0.3306
7,game_mean_speed_raw_median,C_Combined,54,0.3624,0.4415,-0.0412,0.1510
8,game_path_efficiency_median,A_Traditional,54,0.0076,0.0096,-0.1797,-0.3694
9,game_path_efficiency_median,B_Tracking,54,0.0077,0.0099,-0.2451,0.0472



20+ RECEIVER PLAYS: PLAYER-SEASON METRICS


,outcome,model,n_player_seasons,n_players,weighted_mae,weighted_rmse,weighted_r2
0,game_mean_acceleration_raw_median,A_Traditional,93,54,0.1457,0.1804,-0.0811
1,game_mean_acceleration_raw_median,B_Tracking,93,54,0.1495,0.1843,-0.1288
2,game_mean_acceleration_raw_median,Baseline,93,54,0.1418,0.1763,-0.0323
3,game_mean_acceleration_raw_median,C_Combined,93,54,0.1604,0.1967,-0.2855
4,game_mean_speed_raw_median,A_Traditional,93,54,0.3825,0.4755,-0.0423
5,game_mean_speed_raw_median,B_Tracking,93,54,0.3943,0.4927,-0.1187
6,game_mean_speed_raw_median,Baseline,93,54,0.3746,0.4770,-0.0488
7,game_mean_speed_raw_median,C_Combined,93,54,0.3896,0.4740,-0.0355
8,game_path_efficiency_median,A_Traditional,93,54,0.0081,0.0101,-0.1599
9,game_path_efficiency_median,B_Tracking,93,54,0.0084,0.0104,-0.2181


In [36]:

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
RESULTS = ROOT / "reports" / "model_results"

df = pd.read_parquet(
    FEATURES / "wr_movementiq_sensitivity_20plus.parquet"
).copy().reset_index(drop=True)

MODEL_A = ["forty", "vertical", "broad_jump"]

MODEL_B = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
    "combine_coordinate_total_turn_median_across_drills",
    "combine_coordinate_p95_turn_median_across_drills"
]

MODELS = {
    "A_Traditional": MODEL_A,
    "B_Tracking": MODEL_B,
    "C_Combined": MODEL_A + MODEL_B
}

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_total_direction_change_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

assert len(df) == 93
assert df["nfl_id"].nunique() == 54

# Each player's draft cohort must be consistent across seasons
assert df.groupby("nfl_id")["draft_year"].nunique().max() == 1

df["player_weight"] = (
    1 / df.groupby("nfl_id")["nfl_id"].transform("size")
)

logo = LeaveOneGroupOut()
splits = list(
    logo.split(df, groups=df["draft_year"])
)

records = []

for fold_no, (train_idx, test_idx) in enumerate(splits, 1):

    train = df.iloc[train_idx]
    test = df.iloc[test_idx]

    held_out_cohort = int(test["draft_year"].iloc[0])

    train_players = set(train["nfl_id"])
    test_players = set(test["nfl_id"])

    assert train_players.isdisjoint(test_players)

    print(
        f"Fold {fold_no}: held-out draft cohort={held_out_cohort}, "
        f"train players={len(train_players)}, "
        f"test players={len(test_players)}"
    )

    for outcome in OUTCOMES:

        y_train = train[outcome].astype(float)
        y_test = test[outcome].astype(float)

        baseline = np.average(
            y_train,
            weights=train["player_weight"]
        )

        # Baseline predictions
        for idx in test_idx:
            records.append({
                "nfl_id": df.loc[idx, "nfl_id"],
                "draft_year": held_out_cohort,
                "season": df.loc[idx, "season"],
                "fold": fold_no,
                "outcome": outcome,
                "model": "Baseline",
                "actual": df.loc[idx, outcome],
                "predicted": baseline,
                "player_weight": df.loc[idx, "player_weight"]
            })

        for model_name, predictors in MODELS.items():

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=1.0))
            ])

            pipeline.fit(
                train[predictors],
                y_train,
                ridge__sample_weight=train["player_weight"]
            )

            predictions = pipeline.predict(test[predictors])

            for j, idx in enumerate(test_idx):
                records.append({
                    "nfl_id": df.loc[idx, "nfl_id"],
                    "draft_year": held_out_cohort,
                    "season": df.loc[idx, "season"],
                    "fold": fold_no,
                    "outcome": outcome,
                    "model": model_name,
                    "actual": df.loc[idx, outcome],
                    "predicted": predictions[j],
                    "player_weight": df.loc[idx, "player_weight"]
                })

cohort_oof = pd.DataFrame(records)

# --------------------------------------------------
# Player-season metrics
# --------------------------------------------------

season_metrics = []

for (outcome, model), g in cohort_oof.groupby(
    ["outcome", "model"]
):

    season_metrics.append({
        "outcome": outcome,
        "model": model,
        "n_player_seasons": len(g),
        "n_players": g["nfl_id"].nunique(),
        "weighted_mae": mean_absolute_error(
            g["actual"], g["predicted"],
            sample_weight=g["player_weight"]
        ),
        "weighted_rmse": np.sqrt(
            mean_squared_error(
                g["actual"], g["predicted"],
                sample_weight=g["player_weight"]
            )
        ),
        "weighted_r2": r2_score(
            g["actual"], g["predicted"],
            sample_weight=g["player_weight"]
        )
    })

season_metrics = pd.DataFrame(season_metrics)

# --------------------------------------------------
# Player-level metrics
# --------------------------------------------------

player_oof = (
    cohort_oof.groupby(
        ["outcome", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean")
    )
)

player_metrics = []

for (outcome, model), g in player_oof.groupby(
    ["outcome", "model"]
):

    player_metrics.append({
        "outcome": outcome,
        "model": model,
        "n_players": len(g),
        "player_mae": mean_absolute_error(
            g["actual"], g["predicted"]
        ),
        "player_rmse": np.sqrt(
            mean_squared_error(g["actual"], g["predicted"])
        ),
        "player_r2": r2_score(
            g["actual"], g["predicted"]
        ),
        "player_spearman": spearmanr(
            g["actual"], g["predicted"]
        ).statistic
    })

player_metrics = pd.DataFrame(player_metrics)

# --------------------------------------------------
# Per-cohort results: especially important
# --------------------------------------------------

cohort_results = []

for (cohort, outcome, model), g in cohort_oof.groupby(
    ["draft_year", "outcome", "model"]
):

    cohort_results.append({
        "held_out_draft_year": cohort,
        "outcome": outcome,
        "model": model,
        "n_players": g["nfl_id"].nunique(),
        "mae": mean_absolute_error(
            g["actual"], g["predicted"]
        ),
        "rmse": np.sqrt(
            mean_squared_error(g["actual"], g["predicted"])
        )
    })

cohort_metrics = pd.DataFrame(cohort_results)

# Save outputs
cohort_oof.to_parquet(
    RESULTS / "ridge_leave_one_draft_cohort_oof_20plus.parquet",
    index=False
)

cohort_oof.to_csv(
    RESULTS / "ridge_leave_one_draft_cohort_oof_20plus.csv",
    index=False
)

season_metrics.to_csv(
    RESULTS / "ridge_leave_one_draft_cohort_season_metrics_20plus.csv",
    index=False
)

player_metrics.to_csv(
    RESULTS / "ridge_leave_one_draft_cohort_player_metrics_20plus.csv",
    index=False
)

cohort_metrics.to_csv(
    RESULTS / "ridge_leave_one_draft_cohort_by_class_20plus.csv",
    index=False
)

print("\nPLAYER-LEVEL COHORT VALIDATION")
display(player_metrics.round(4))

print("\nPLAYER-SEASON COHORT VALIDATION")
display(season_metrics.round(4))

print("\nRESULTS BY HELD-OUT DRAFT COHORT")
display(cohort_metrics.round(4))

print("\nAll cohort validation outputs saved.")


Fold 1: held-out draft cohort=2023, train players=31, test players=23
Fold 2: held-out draft cohort=2024, train players=37, test players=17
Fold 3: held-out draft cohort=2025, train players=40, test players=14

PLAYER-LEVEL COHORT VALIDATION


,outcome,model,n_players,player_mae,player_rmse,player_r2,player_spearman
0,game_mean_acceleration_raw_median,A_Traditional,54,0.1408,0.1711,-0.1963,-0.1094
1,game_mean_acceleration_raw_median,B_Tracking,54,0.1390,0.1675,-0.1465,-0.0764
2,game_mean_acceleration_raw_median,Baseline,54,0.1342,0.1646,-0.1069,-0.2774
3,game_mean_acceleration_raw_median,C_Combined,54,0.1566,0.1889,-0.4580,-0.0861
4,game_mean_speed_raw_median,A_Traditional,54,0.3822,0.4682,-0.1710,0.0298
5,game_mean_speed_raw_median,B_Tracking,54,0.3833,0.5135,-0.4086,-0.0310
6,game_mean_speed_raw_median,Baseline,54,0.3372,0.4397,-0.0329,-0.1196
7,game_mean_speed_raw_median,C_Combined,54,0.4478,0.5595,-0.6719,-0.0089
8,game_path_efficiency_median,A_Traditional,54,0.0086,0.0111,-0.5689,-0.2461
9,game_path_efficiency_median,B_Tracking,54,0.0087,0.0112,-0.6113,0.1184



PLAYER-SEASON COHORT VALIDATION


,outcome,model,n_player_seasons,n_players,weighted_mae,weighted_rmse,weighted_r2
0,game_mean_acceleration_raw_median,A_Traditional,93,54,0.1505,0.1868,-0.1595
1,game_mean_acceleration_raw_median,B_Tracking,93,54,0.1478,0.1835,-0.1191
2,game_mean_acceleration_raw_median,Baseline,93,54,0.1442,0.1809,-0.0869
3,game_mean_acceleration_raw_median,C_Combined,93,54,0.1656,0.2032,-0.3722
4,game_mean_speed_raw_median,A_Traditional,93,54,0.4047,0.4990,-0.1476
5,game_mean_speed_raw_median,B_Tracking,93,54,0.4168,0.5417,-0.3526
6,game_mean_speed_raw_median,Baseline,93,54,0.3689,0.4724,-0.0284
7,game_mean_speed_raw_median,C_Combined,93,54,0.4707,0.5854,-0.5798
8,game_path_efficiency_median,A_Traditional,93,54,0.0091,0.0115,-0.5063
9,game_path_efficiency_median,B_Tracking,93,54,0.0093,0.0117,-0.5440



RESULTS BY HELD-OUT DRAFT COHORT


,held_out_draft_year,outcome,model,n_players,mae,rmse
0,2023,game_mean_acceleration_raw_median,A_Traditional,23,0.1765,0.2177
1,2023,game_mean_acceleration_raw_median,B_Tracking,23,0.1577,0.2024
2,2023,game_mean_acceleration_raw_median,Baseline,23,0.1643,0.2057
3,2023,game_mean_acceleration_raw_median,C_Combined,23,0.1956,0.2397
4,2023,game_mean_speed_raw_median,A_Traditional,23,0.4738,0.5752
...,...,...,...,...,...,...
67,2025,game_separation_at_pass_forward_median,C_Combined,14,0.8206,1.0273
68,2025,game_total_direction_change_raw_median,A_Traditional,14,10.6395,13.4991
69,2025,game_total_direction_change_raw_median,B_Tracking,14,11.8606,14.2610
70,2025,game_total_direction_change_raw_median,Baseline,14,11.2598,13.8801



All cohort validation outputs saved.


In [37]:

import pandas as pd
import numpy as np

ROOT = "/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027"
RESULTS = f"{ROOT}/reports/model_results"

cohort = pd.read_csv(
    f"{RESULTS}/ridge_leave_one_draft_cohort_by_class_20plus.csv"
)

peak = cohort[
    cohort["outcome"] == "game_peak_speed_raw_median"
].copy()

print("=" * 85)
print("PEAK SPEED: PERFORMANCE BY HELD-OUT DRAFT COHORT")
print("=" * 85)

display(
    peak.sort_values(
        ["held_out_draft_year", "mae"]
    ).round(4)
)

# Pivot MAE by model for each cohort
pivot = peak.pivot(
    index="held_out_draft_year",
    columns="model",
    values="mae"
)

pivot["traditional_improvement_pct"] = (
    (pivot["Baseline"] - pivot["A_Traditional"])
    / pivot["Baseline"] * 100
)

pivot["tracking_improvement_pct"] = (
    (pivot["Baseline"] - pivot["B_Tracking"])
    / pivot["Baseline"] * 100
)

pivot["combined_improvement_pct"] = (
    (pivot["Baseline"] - pivot["C_Combined"])
    / pivot["Baseline"] * 100
)

pivot["combined_vs_traditional_pct"] = (
    (pivot["A_Traditional"] - pivot["C_Combined"])
    / pivot["A_Traditional"] * 100
)

print("\nMAE IMPROVEMENT BY HELD-OUT COHORT (%)")
display(pivot.round(3))

pivot.to_csv(
    f"{RESULTS}/peak_speed_cohort_comparison_20plus.csv"
)

print("\nSaved peak-speed cohort comparison.")


PEAK SPEED: PERFORMANCE BY HELD-OUT DRAFT COHORT


,held_out_draft_year,outcome,model,n_players,mae,rmse
14,2023,game_peak_speed_raw_median,Baseline,23,0.4616,0.6290
13,2023,game_peak_speed_raw_median,B_Tracking,23,0.4647,0.6325
15,2023,game_peak_speed_raw_median,C_Combined,23,0.4735,0.6051
12,2023,game_peak_speed_raw_median,A_Traditional,23,0.4952,0.6274
37,2024,game_peak_speed_raw_median,B_Tracking,17,0.3904,0.5534
39,2024,game_peak_speed_raw_median,C_Combined,17,0.4055,0.5134
38,2024,game_peak_speed_raw_median,Baseline,17,0.4534,0.5875
36,2024,game_peak_speed_raw_median,A_Traditional,17,0.4592,0.5464
60,2025,game_peak_speed_raw_median,A_Traditional,14,0.2851,0.3433
62,2025,game_peak_speed_raw_median,Baseline,14,0.3776,0.4346



MAE IMPROVEMENT BY HELD-OUT COHORT (%)


model,A_Traditional,B_Tracking,Baseline,C_Combined,traditional_improvement_pct,tracking_improvement_pct,combined_improvement_pct,combined_vs_traditional_pct
held_out_draft_year,,,,,,,,
2023,0.495,0.465,0.462,0.474,-7.281,-0.672,-2.588,4.374
2024,0.459,0.390,0.453,0.405,-1.271,13.897,10.570,11.693
2025,0.285,0.506,0.378,0.433,24.484,-34.015,-14.731,-51.930



Saved peak-speed cohort comparison.


In [38]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(
    FEATURES / "wr_movementiq_sensitivity_20plus.parquet"
).copy()

OUTCOME = "game_peak_speed_raw_median"

PREDICTORS = [
    "forty",
    "vertical",
    "broad_jump",
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
    "combine_coordinate_total_turn_median_across_drills",
    "combine_coordinate_p95_turn_median_across_drills"
]

# --------------------------------------------------
# 1. Outcome distribution by draft cohort
# --------------------------------------------------

outcome_summary = (
    df.groupby("draft_year")[OUTCOME]
    .agg(
        n="count",
        mean="mean",
        std="std",
        median="median",
        minimum="min",
        p25=lambda x: x.quantile(.25),
        p75=lambda x: x.quantile(.75),
        maximum="max"
    )
    .reset_index()
)

print("=" * 80)
print("PEAK SPEED DISTRIBUTION BY DRAFT COHORT")
print("=" * 80)
display(outcome_summary.round(4))

# --------------------------------------------------
# 2. Predictor distributions by cohort
# --------------------------------------------------

predictor_summary = (
    df.groupby("draft_year")[PREDICTORS]
    .agg(["mean", "std", "median"])
)

print("\nPREDICTOR DISTRIBUTIONS BY COHORT")
display(predictor_summary.round(4))

# --------------------------------------------------
# 3. Player counts and exposure
# --------------------------------------------------

exposure_summary = (
    df.groupby("draft_year")
    .agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique"),
        median_receiver_plays=("receiver_plays", "median"),
        mean_receiver_plays=("receiver_plays", "mean"),
        min_receiver_plays=("receiver_plays", "min"),
        max_receiver_plays=("receiver_plays", "max")
    )
    .reset_index()
)

print("\nCOHORT EXPOSURE")
display(exposure_summary.round(2))

# --------------------------------------------------
# 4. Predictor missingness by cohort
# --------------------------------------------------

missingness = (
    df.groupby("draft_year")[PREDICTORS]
    .apply(lambda x: x.isna().mean() * 100)
)

print("\nPREDICTOR MISSINGNESS (%)")
display(missingness.round(2))

# --------------------------------------------------
# 5. Save audit outputs
# --------------------------------------------------

outcome_summary.to_csv(
    REPORTS / "peak_speed_distribution_by_draft_cohort.csv",
    index=False
)

predictor_summary.to_csv(
    REPORTS / "predictor_distribution_by_draft_cohort.csv"
)

exposure_summary.to_csv(
    REPORTS / "exposure_by_draft_cohort_20plus.csv",
    index=False
)

missingness.to_csv(
    REPORTS / "predictor_missingness_by_draft_cohort.csv"
)

print("\nCOHORT SHIFT AUDIT SAVED SUCCESSFULLY.")


PEAK SPEED DISTRIBUTION BY DRAFT COHORT


,draft_year,n,mean,std,median,minimum,p25,p75,maximum
0,2023,51,6.7969,0.6336,6.8250,4.795,6.550,7.1800,8.350
1,2024,28,6.7155,0.5871,6.7925,5.210,6.465,7.1425,7.385
2,2025,14,6.8489,0.4403,6.8750,5.980,6.610,7.1975,7.400



PREDICTOR DISTRIBUTIONS BY COHORT


forty                vertical                broad_jump          \
              mean     std median     mean     std median       mean     std   
draft_year                                                                     
2023        4.4780  0.0792  4.490  37.2442  2.9668   37.5   126.2889  5.0929   
2024        4.4496  0.0854  4.445  39.0370  2.1879   39.0   126.8148  3.4756   
2025        4.4323  0.0929  4.410  37.6250  2.3944   38.0   127.2500  4.0704   

                  combine_mean_speed_median_across_drills  ...  \
           median                                    mean  ...   
draft_year                                                 ...   
2023        128.0                                  5.7594  ...   
2024        126.0                                  5.8512  ...   
2025        127.5                                  5.2966  ...   

           combine_mean_acceleration_median_across_drills  \
                                                   median   
draft_year                                                  
2023                                               3.2506   
2024                                               3.3639   
2025                                               3.4488   

           combine_path_efficiency_median_across_drills                  \
                                                   mean     std  median   
draft_year                                                                
2023                                             0.8963  0.0143  0.8937   
2024                                             0.8945  0.0188  0.8996   
2025                                             0.8998  0.0262  0.8977   

           combine_coordinate_total_turn_median_across_drills           \
                                                         mean      std   
draft_year                                                               
2023                                                 181.3700  15.2763   
2024                                                 188.2496  13.7692   
2025                                                 122.0513  22.2953   

                     combine_coordinate_p95_turn_median_across_drills          \
              median                                             mean     std   
draft_year                                                                      
2023        181.4596                                           8.0554  0.6728   
2024        191.1844                                           8.3167  0.6995   
2025        119.8082                                           5.7363  0.5022   

                    
            median  
draft_year          
2023        8.1301  
2024        8.3395  
2025        5.7744  

[3 rows x 27 columns]


COHORT EXPOSURE


,draft_year,player_seasons,unique_players,median_receiver_plays,mean_receiver_plays,min_receiver_plays,max_receiver_plays
0,2023,51,23,66.0,66.57,22,156
1,2024,28,17,54.5,61.79,21,132
2,2025,14,14,37.0,43.29,21,87



PREDICTOR MISSINGNESS (%)


,forty,vertical,broad_jump,combine_mean_speed_median_across_drills,combine_peak_speed_median_across_drills,combine_mean_acceleration_median_across_drills,combine_path_efficiency_median_across_drills,combine_coordinate_total_turn_median_across_drills,combine_coordinate_p95_turn_median_across_drills
draft_year,,,,,,,,,
2023,19.61,15.69,11.76,0.0,0.0,0.0,0.0,0.0,0.0
2024,0.00,3.57,3.57,0.0,0.0,0.0,0.0,0.0,0.0
2025,7.14,14.29,14.29,0.0,0.0,0.0,0.0,0.0,0.0



COHORT SHIFT AUDIT SAVED SUCCESSFULLY.


In [39]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

REPORTS.mkdir(parents=True, exist_ok=True)

path = FEATURES / "wr_combine_attempt_features_enriched.parquet"
attempts = pd.read_parquet(path)

print("DATASET SHAPE:", attempts.shape)
print("\nAVAILABLE COLUMNS:")
print(attempts.columns.tolist())

# Primary drills used in the research
PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

# Restrict to primary drills
d = attempts[
    attempts["drill_type"].isin(PRIMARY_DRILLS)
].copy()

# --------------------------------------------------
# 1. Drill coverage by draft cohort
# --------------------------------------------------

coverage = (
    d.groupby(["draft_year", "drill_type"])
    .agg(
        attempts=("nfl_id", "size"),
        players=("nfl_id", "nunique")
    )
    .reset_index()
)

cohort_players = (
    d.groupby("draft_year")["nfl_id"]
    .nunique()
    .rename("cohort_players")
)

coverage = coverage.merge(
    cohort_players,
    on="draft_year",
    how="left"
)

coverage["player_coverage_pct"] = (
    coverage["players"] / coverage["cohort_players"] * 100
)

print("\n1. PRIMARY DRILL COVERAGE BY COHORT")
display(
    coverage.sort_values(
        ["draft_year", "drill_type"]
    ).round(2)
)

# --------------------------------------------------
# 2. Number of primary drills per player
# --------------------------------------------------

player_drill_count = (
    d.groupby(["draft_year", "nfl_id"])["drill_type"]
    .nunique()
    .reset_index(name="primary_drills_observed")
)

drill_count_summary = (
    player_drill_count.groupby("draft_year")[
        "primary_drills_observed"
    ]
    .agg(
        players="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

print("\n2. PRIMARY DRILLS PER PLAYER")
display(drill_count_summary.round(2))

# --------------------------------------------------
# 3. Inspect movement feature names
# --------------------------------------------------

feature_candidates = [
    "mean_speed",
    "peak_speed",
    "p95_speed",
    "mean_acceleration",
    "peak_acceleration",
    "coordinate_path_length",
    "net_displacement",
    "path_efficiency",
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "turning_rate_degrees_per_second"
]

available_features = [
    c for c in feature_candidates
    if c in d.columns
]

print("\n3. AVAILABLE MOVEMENT FEATURES:")
print(available_features)

# --------------------------------------------------
# 4. Feature distributions within each drill/cohort
# --------------------------------------------------

if available_features:

    drill_feature_summary = (
        d.groupby(["draft_year", "drill_type"])[available_features]
        .median()
        .reset_index()
    )

    print("\n4. WITHIN-DRILL FEATURE MEDIANS")
    display(drill_feature_summary.round(3))

    drill_feature_summary.to_csv(
        REPORTS / "movement_features_by_drill_and_cohort.csv",
        index=False
    )

# --------------------------------------------------
# 5. Save coverage outputs
# --------------------------------------------------

coverage.to_csv(
    REPORTS / "primary_drill_coverage_by_cohort.csv",
    index=False
)

player_drill_count.to_csv(
    REPORTS / "primary_drill_count_per_player.csv",
    index=False
)

drill_count_summary.to_csv(
    REPORTS / "primary_drill_count_summary_by_cohort.csv",
    index=False
)

print("\nDRILL COMPOSITION AUDIT SAVED.")


DATASET SHAPE: (1771, 42)

AVAILABLE COLUMNS:
['draft_year', 'nfl_id', 'event_id', 'drill_type', 'drill_name', 'attempt', 'observations', 'start_time', 'end_time', 'duration_seconds', 'start_x', 'start_y', 'end_x', 'end_y', 'mean_speed', 'peak_speed', 'p95_speed', 'speed_std', 'mean_acceleration', 'peak_acceleration', 'minimum_acceleration', 'acceleration_std', 'coordinate_path_length', 'summed_recorded_displacement', 'total_direction_change', 'median_sampling_interval', 'maximum_sampling_interval', 'large_sampling_gaps', 'net_displacement', 'path_efficiency', 'valid_turn_pairs', 'coordinate_total_turn', 'coordinate_mean_turn', 'coordinate_median_turn', 'coordinate_p95_turn', 'coordinate_max_turn', 'recorded_total_turn', 'recorded_mean_turn', 'speed_threshold', 'turning_rate_degrees_per_second', 'valid_pair_coverage', 'turning_coverage_flag']

1. PRIMARY DRILL COVERAGE BY COHORT


,draft_year,drill_type,attempts,players,cohort_players,player_coverage_pct



2. PRIMARY DRILLS PER PLAYER


,draft_year,players,mean,median,minimum,maximum



3. AVAILABLE MOVEMENT FEATURES:
['mean_speed', 'peak_speed', 'p95_speed', 'mean_acceleration', 'peak_acceleration', 'coordinate_path_length', 'net_displacement', 'path_efficiency', 'coordinate_total_turn', 'coordinate_mean_turn', 'coordinate_p95_turn', 'turning_rate_degrees_per_second']

4. WITHIN-DRILL FEATURE MEDIANS


,draft_year,drill_type,mean_speed,peak_speed,p95_speed,mean_acceleration,peak_acceleration,coordinate_path_length,net_displacement,path_efficiency,coordinate_total_turn,coordinate_mean_turn,coordinate_p95_turn,turning_rate_degrees_per_second



DRILL COMPOSITION AUDIT SAVED.


In [40]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"

attempts = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

print("=" * 75)
print("DRILL LABEL DIAGNOSTICS")
print("=" * 75)

print("\n1. UNIQUE drill_type VALUES")
print("Count:", attempts["drill_type"].nunique())
print(attempts["drill_type"].value_counts(dropna=False).to_string())

print("\n2. UNIQUE drill_name VALUES")
print("Count:", attempts["drill_name"].nunique())
print(attempts["drill_name"].value_counts(dropna=False).to_string())

print("\n3. SAMPLE DRILL TYPE / NAME PAIRS")
display(
    attempts[
        ["draft_year", "drill_type", "drill_name", "attempt"]
    ].drop_duplicates().head(30)
)

print("\n4. PRIMARY DRILL MATCH TEST")

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

for column in ["drill_type", "drill_name"]:
    values = set(attempts[column].dropna().astype(str))
    matches = sorted(values.intersection(PRIMARY_DRILLS))

    print(f"\n{column}: {len(matches)} matching drills")
    print(matches)

print("\n5. DRAFT COHORT COUNTS")
print(
    attempts.groupby("draft_year")["nfl_id"]
    .nunique()
    .to_string()
)


DRILL LABEL DIAGNOSTICS

1. UNIQUE drill_type VALUES
Count: 4
drill_type
SKILL_DRILLS_WR     1529
FORTY_YARD_DASH      176
SHORT_SHUTTLE         36
THREE_CONE_DRILL      30

2. UNIQUE drill_name VALUES
Count: 18
drill_name
GAUNTLET_DRILL             200
FORTY_YARD_DASH            176
DAGGER_ROUTE_LEFT          115
RED_ZONE_FADE_RIGHT        112
POST_CORNER_ROUTE_RIGHT    111
SLANT_ROUTE_LEFT           110
COMEBACK_ROUTE_RIGHT       110
CURL_ROUTE_RIGHT           110
SLOT_SAIL_ROUTE_RIGHT      106
GO_ROUTE_RIGHT             105
SLOT_STRIKE_ROUTE_LEFT     102
OVER_SHOULDER_ADJUST       101
SPEED_OUT_ROUTE_LEFT        87
POST_CORNER_ROUTE_LEFT      80
CIRCUS_ROUTE_LEFT           43
SPEED_OUT_LEFT              37
SHORT_SHUTTLE               36
THREE_CONE_DRILL            30

3. SAMPLE DRILL TYPE / NAME PAIRS


,draft_year,drill_type,drill_name,attempt
0,2023,SKILL_DRILLS_WR,SPEED_OUT_ROUTE_LEFT,1
1,2023,FORTY_YARD_DASH,FORTY_YARD_DASH,1
2,2023,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,1
3,2023,SKILL_DRILLS_WR,GAUNTLET_DRILL,2
4,2023,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,1
5,2023,SKILL_DRILLS_WR,POST_CORNER_ROUTE_LEFT,1
6,2023,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,1
7,2023,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,1
9,2023,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,1
11,2023,SKILL_DRILLS_WR,SPEED_OUT_ROUTE_LEFT,2



4. PRIMARY DRILL MATCH TEST

drill_type: 0 matching drills
[]

drill_name: 11 matching drills
['COMEBACK_ROUTE_RIGHT', 'CURL_ROUTE_RIGHT', 'DAGGER_ROUTE_LEFT', 'GAUNTLET_DRILL', 'GO_ROUTE_RIGHT', 'OVER_SHOULDER_ADJUST', 'POST_CORNER_ROUTE_RIGHT', 'RED_ZONE_FADE_RIGHT', 'SLANT_ROUTE_LEFT', 'SLOT_SAIL_ROUTE_RIGHT', 'SLOT_STRIKE_ROUTE_LEFT']

5. DRAFT COHORT COUNTS
draft_year
2023    39
2024    31
2025    37


In [41]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

REPORTS.mkdir(parents=True, exist_ok=True)

attempts = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

FEATURES_TO_COMPARE = [
    "mean_speed",
    "peak_speed",
    "p95_speed",
    "mean_acceleration",
    "coordinate_path_length",
    "net_displacement",
    "path_efficiency",
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "turning_rate_degrees_per_second"
]

# Correct filter: individual drill names
d = attempts[
    attempts["drill_name"].isin(PRIMARY_DRILLS)
].copy()

assert d["drill_name"].nunique() == 11

print("PRIMARY DRILL ATTEMPTS:", len(d))
print("UNIQUE PLAYERS:", d["nfl_id"].nunique())
print("DRAFT COHORTS:", sorted(d["draft_year"].unique()))

# --------------------------------------------------
# 1. Drill coverage by cohort
# --------------------------------------------------

coverage = (
    d.groupby(["draft_year", "drill_name"])
    .agg(
        attempts=("nfl_id", "size"),
        players=("nfl_id", "nunique")
    )
    .reset_index()
)

cohort_players = (
    d.groupby("draft_year")["nfl_id"]
    .nunique()
    .rename("cohort_players")
)

coverage = coverage.merge(
    cohort_players,
    on="draft_year",
    how="left"
)

coverage["player_coverage_pct"] = (
    coverage["players"] /
    coverage["cohort_players"] * 100
)

print("\n1. DRILL COVERAGE BY COHORT")
display(
    coverage.sort_values(
        ["drill_name", "draft_year"]
    ).round(2)
)

# --------------------------------------------------
# 2. Number of drills per player
# --------------------------------------------------

player_drills = (
    d.groupby(["draft_year", "nfl_id"])["drill_name"]
    .nunique()
    .reset_index(name="primary_drills_observed")
)

player_drill_summary = (
    player_drills.groupby("draft_year")[
        "primary_drills_observed"
    ]
    .agg(
        players="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

print("\n2. PRIMARY DRILLS PER PLAYER")
display(player_drill_summary.round(2))

# --------------------------------------------------
# 3. Within-drill movement feature medians
# --------------------------------------------------

available_features = [
    c for c in FEATURES_TO_COMPARE
    if c in d.columns
]

within_drill = (
    d.groupby(["draft_year", "drill_name"])[available_features]
    .median()
    .reset_index()
)

print("\n3. WITHIN-DRILL FEATURE MEDIANS")
display(within_drill.round(3))

# --------------------------------------------------
# 4. Compare cohort medians within each drill
# --------------------------------------------------

turning_comparison = within_drill.pivot(
    index="drill_name",
    columns="draft_year",
    values="coordinate_total_turn"
)

turning_comparison.columns = [
    f"total_turn_{year}" for year in turning_comparison.columns
]

print("\n4. TOTAL TURN BY DRILL AND COHORT")
display(turning_comparison.round(2))

# --------------------------------------------------
# 5. Save all outputs
# --------------------------------------------------

coverage.to_csv(
    REPORTS / "primary_drill_coverage_by_cohort_v2.csv",
    index=False
)

player_drills.to_csv(
    REPORTS / "primary_drill_count_per_player_v2.csv",
    index=False
)

player_drill_summary.to_csv(
    REPORTS / "primary_drill_count_summary_by_cohort_v2.csv",
    index=False
)

within_drill.to_csv(
    REPORTS / "movement_features_by_drill_and_cohort_v2.csv",
    index=False
)

turning_comparison.to_csv(
    REPORTS / "total_turn_by_drill_and_cohort.csv"
)

print("\nCORRECTED DRILL AUDIT SAVED SUCCESSFULLY.")


PRIMARY DRILL ATTEMPTS: 1282
UNIQUE PLAYERS: 107
DRAFT COHORTS: [np.int64(2023), np.int64(2024), np.int64(2025)]

1. DRILL COVERAGE BY COHORT


,draft_year,drill_name,attempts,players,cohort_players,player_coverage_pct
0,2023,COMEBACK_ROUTE_RIGHT,37,36,39,92.31
11,2024,COMEBACK_ROUTE_RIGHT,33,28,31,90.32
22,2025,COMEBACK_ROUTE_RIGHT,40,34,37,91.89
1,2023,CURL_ROUTE_RIGHT,43,37,39,94.87
12,2024,CURL_ROUTE_RIGHT,31,30,31,96.77
23,2025,CURL_ROUTE_RIGHT,36,36,37,97.30
2,2023,DAGGER_ROUTE_LEFT,43,39,39,100.00
13,2024,DAGGER_ROUTE_LEFT,32,31,31,100.00
24,2025,DAGGER_ROUTE_LEFT,40,37,37,100.00
3,2023,GAUNTLET_DRILL,71,39,39,100.00



2. PRIMARY DRILLS PER PLAYER


,draft_year,players,mean,median,minimum,maximum
0,2023,39,10.36,11.0,5,11
1,2024,31,10.48,11.0,5,11
2,2025,37,10.38,11.0,4,11



3. WITHIN-DRILL FEATURE MEDIANS


,draft_year,drill_name,mean_speed,peak_speed,p95_speed,mean_acceleration,coordinate_path_length,net_displacement,path_efficiency,coordinate_total_turn,coordinate_mean_turn,coordinate_p95_turn,turning_rate_degrees_per_second
0,2023,COMEBACK_ROUTE_RIGHT,5.042,8.227,8.122,3.945,26.915,18.029,0.673,200.617,4.026,13.472,38.580
1,2023,CURL_ROUTE_RIGHT,4.025,7.060,6.900,4.346,21.097,16.299,0.773,303.750,5.901,17.883,56.421
2,2023,DAGGER_ROUTE_LEFT,6.254,8.170,8.089,3.802,40.613,35.405,0.886,196.918,3.257,7.879,31.286
3,2023,GAUNTLET_DRILL,6.781,8.760,8.709,2.397,60.818,54.292,0.892,199.298,2.350,7.993,22.441
4,2023,GO_ROUTE_RIGHT,7.766,10.290,10.261,1.990,56.562,56.246,0.995,117.211,1.701,4.764,16.029
5,2023,OVER_SHOULDER_ADJUST,5.841,7.573,7.522,1.442,36.039,35.885,0.996,130.467,2.126,6.340,20.861
6,2023,POST_CORNER_ROUTE_RIGHT,7.069,9.302,9.245,3.012,45.248,42.005,0.923,159.201,2.548,6.334,24.707
7,2023,RED_ZONE_FADE_RIGHT,3.909,6.460,6.436,2.044,17.793,17.585,0.989,140.856,3.474,11.484,30.152
8,2023,SLANT_ROUTE_LEFT,5.230,8.593,8.425,3.136,23.936,22.913,0.960,123.660,3.232,8.033,27.046
9,2023,SLOT_SAIL_ROUTE_RIGHT,6.078,8.312,8.210,3.982,44.908,34.740,0.760,229.179,3.416,7.979,30.213



4. TOTAL TURN BY DRILL AND COHORT


,total_turn_2023,total_turn_2024,total_turn_2025
drill_name,,,
COMEBACK_ROUTE_RIGHT,200.62,201.32,187.42
CURL_ROUTE_RIGHT,303.75,319.88,261.75
DAGGER_ROUTE_LEFT,196.92,199.79,147.67
GAUNTLET_DRILL,199.30,194.37,117.00
GO_ROUTE_RIGHT,117.21,119.59,40.87
OVER_SHOULDER_ADJUST,130.47,108.97,40.06
POST_CORNER_ROUTE_RIGHT,159.20,182.57,88.47
RED_ZONE_FADE_RIGHT,140.86,142.12,62.14
SLANT_ROUTE_LEFT,123.66,125.13,70.36



CORRECTED DRILL AUDIT SAVED SUCCESSFULLY.


In [42]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

RAW = ROOT / "data" / "raw" / "nfl-big-data-bowl-2027"
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

REPORTS.mkdir(parents=True, exist_ok=True)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

# --------------------------------------------------
# 1. Load enriched attempt-level features
# --------------------------------------------------

attempts = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

d = attempts[
    attempts["drill_name"].isin(PRIMARY_DRILLS)
].copy()

print("PRIMARY ATTEMPTS:", len(d))
print("DRAFT COHORTS:", sorted(d["draft_year"].unique()))

# --------------------------------------------------
# 2. Attempt quality comparison by cohort/drill
# --------------------------------------------------

quality_columns = [
    "observations",
    "duration_seconds",
    "median_sampling_interval",
    "maximum_sampling_interval",
    "large_sampling_gaps",
    "valid_turn_pairs",
    "valid_pair_coverage",
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "recorded_total_turn",
    "recorded_mean_turn",
    "mean_speed",
    "peak_speed",
    "path_efficiency",
    "speed_threshold"
]

available_quality = [
    c for c in quality_columns if c in d.columns
]

quality_summary = (
    d.groupby(["draft_year", "drill_name"])[available_quality]
    .median()
    .reset_index()
)

print("\n1. ATTEMPT QUALITY BY COHORT AND DRILL")
display(quality_summary.round(3))

# --------------------------------------------------
# 3. Confirm speed threshold consistency
# --------------------------------------------------

print("\n2. SPEED THRESHOLD COUNTS")
print(
    d.groupby(["draft_year", "speed_threshold"])
    .size()
    .rename("attempts")
    .reset_index()
    .to_string(index=False)
)

# --------------------------------------------------
# 4. Load raw Combine tracking
# --------------------------------------------------

tracking_path = RAW / "combine_tracking.csv"

tracking = pd.read_csv(
    tracking_path,
    low_memory=False
)

# Only players and WRs
combine_results = pd.read_csv(
    RAW / "combine_results.csv",
    low_memory=False
)

wr_ids = set(
    combine_results.loc[
        combine_results["combine_position"].astype(str).str.upper() == "WR",
        "nfl_id"
    ].dropna()
)

tracking = tracking[
    (tracking["entity_type"] == "PLAYER") &
    (tracking["nfl_id"].isin(wr_ids)) &
    (tracking["drill_name"].isin(PRIMARY_DRILLS))
].copy()

tracking["time"] = pd.to_datetime(
    tracking["time"],
    errors="coerce"
)

for col in ["x", "y", "s", "dir"]:
    tracking[col] = pd.to_numeric(
        tracking[col],
        errors="coerce"
    )

tracking = tracking.dropna(
    subset=["time", "x", "y"]
)

tracking = tracking.sort_values(
    ["draft_year", "nfl_id", "drill_name", "attempt", "time"]
)

# --------------------------------------------------
# 5. Derive coordinate headings
# --------------------------------------------------

group_cols = [
    "draft_year",
    "nfl_id",
    "drill_name",
    "attempt",
    "event_id"
]

tracking["dt"] = (
    tracking.groupby(group_cols)["time"]
    .diff()
    .dt.total_seconds()
)

tracking["dx"] = (
    tracking.groupby(group_cols)["x"].diff()
)

tracking["dy"] = (
    tracking.groupby(group_cols)["y"].diff()
)

tracking["coordinate_speed"] = (
    np.sqrt(tracking["dx"]**2 + tracking["dy"]**2)
    / tracking["dt"]
)

tracking["coordinate_heading"] = (
    np.degrees(
        np.arctan2(tracking["dx"], tracking["dy"])
    ) % 360
)

# Circular angular difference
def angular_difference(a, b):
    return np.abs((a - b + 180) % 360 - 180)

tracking["heading_error_current"] = angular_difference(
    tracking["coordinate_heading"],
    tracking["dir"]
)

tracking["previous_dir"] = (
    tracking.groupby(group_cols)["dir"].shift(1)
)

tracking["heading_error_previous"] = angular_difference(
    tracking["coordinate_heading"],
    tracking["previous_dir"]
)

# Use reliable movement observations for heading validation
moving = tracking[
    (tracking["coordinate_speed"] > 3) &
    (tracking["dt"] > 0) &
    (tracking["dir"].notna())
].copy()

# --------------------------------------------------
# 6. Heading agreement by cohort and drill
# --------------------------------------------------

heading_summary = (
    moving.groupby(["draft_year", "drill_name"])
    .agg(
        observations=("heading_error_current", "count"),
        median_coordinate_speed=("coordinate_speed", "median"),
        median_recorded_speed=("s", "median"),
        median_heading_error_current=("heading_error_current", "median"),
        p90_heading_error_current=(
            "heading_error_current",
            lambda x: x.quantile(.90)
        ),
        median_heading_error_previous=("heading_error_previous", "median")
    )
    .reset_index()
)

print("\n3. COORDINATE VS RECORDED HEADING")
display(heading_summary.round(3))

# --------------------------------------------------
# 7. Raw tracking completeness by cohort
# --------------------------------------------------

raw_quality = (
    tracking.groupby("draft_year")
    .agg(
        tracking_rows=("nfl_id", "size"),
        players=("nfl_id", "nunique"),
        missing_speed=("s", lambda x: x.isna().sum()),
        missing_direction=("dir", lambda x: x.isna().sum()),
        missing_coordinates=("x", lambda x: x.isna().sum()),
        median_speed=("s", "median")
    )
    .reset_index()
)

print("\n4. RAW TRACKING COMPLETENESS")
display(raw_quality.round(3))

# --------------------------------------------------
# 8. Save audit results
# --------------------------------------------------

quality_summary.to_csv(
    REPORTS / "turning_attempt_quality_by_cohort_drill.csv",
    index=False
)

heading_summary.to_csv(
    REPORTS / "coordinate_heading_validation_by_cohort_drill.csv",
    index=False
)

raw_quality.to_csv(
    REPORTS / "raw_tracking_completeness_by_cohort.csv",
    index=False
)

print("\nTRACKING CONSISTENCY AUDIT SAVED.")



PRIMARY ATTEMPTS: 1282
DRAFT COHORTS: [np.int64(2023), np.int64(2024), np.int64(2025)]

1. ATTEMPT QUALITY BY COHORT AND DRILL


,draft_year,drill_name,observations,duration_seconds,median_sampling_interval,maximum_sampling_interval,large_sampling_gaps,valid_turn_pairs,valid_pair_coverage,coordinate_total_turn,coordinate_mean_turn,coordinate_p95_turn,recorded_total_turn,recorded_mean_turn,mean_speed,peak_speed,path_efficiency,speed_threshold
0,2023,COMEBACK_ROUTE_RIGHT,53.0,5.20,0.1,0.1,0.0,49.0,0.945,200.617,4.026,13.472,137.257,2.834,5.042,8.227,0.673,1.0
1,2023,CURL_ROUTE_RIGHT,54.0,5.30,0.1,0.1,0.0,51.0,0.963,303.750,5.901,17.883,242.854,4.643,4.025,7.060,0.773,1.0
2,2023,DAGGER_ROUTE_LEFT,65.0,6.40,0.1,0.1,0.0,62.0,0.962,196.918,3.257,7.879,148.428,2.420,6.254,8.170,0.886,1.0
3,2023,GAUNTLET_DRILL,90.0,8.90,0.1,0.1,0.0,85.0,0.955,199.298,2.350,7.993,115.119,1.326,6.781,8.760,0.892,1.0
4,2023,GO_ROUTE_RIGHT,73.0,7.20,0.1,0.1,0.0,69.0,0.958,117.211,1.701,4.764,39.575,0.547,7.766,10.290,0.995,1.0
5,2023,OVER_SHOULDER_ADJUST,62.0,6.10,0.1,0.1,0.0,59.5,0.982,130.467,2.126,6.340,25.222,0.423,5.841,7.573,0.996,1.0
6,2023,POST_CORNER_ROUTE_RIGHT,65.0,6.40,0.1,0.1,0.0,62.0,0.969,159.201,2.548,6.334,109.062,1.764,7.069,9.302,0.923,1.0
7,2023,RED_ZONE_FADE_RIGHT,45.0,4.40,0.1,0.1,0.0,39.0,0.886,140.856,3.474,11.484,63.146,1.543,3.909,6.460,0.989,1.0
8,2023,SLANT_ROUTE_LEFT,46.0,4.50,0.1,0.1,0.0,39.0,0.844,123.660,3.232,8.033,93.170,2.369,5.230,8.593,0.960,1.0
9,2023,SLOT_SAIL_ROUTE_RIGHT,74.0,7.30,0.1,0.1,0.0,64.5,0.888,229.179,3.416,7.979,182.008,2.749,6.078,8.312,0.760,1.0



2. SPEED THRESHOLD COUNTS
 draft_year  speed_threshold  attempts
       2023              1.0       468
       2024              1.0       377
       2025              1.0       437

3. COORDINATE VS RECORDED HEADING


,draft_year,drill_name,observations,median_coordinate_speed,median_recorded_speed,median_heading_error_current,p90_heading_error_current,median_heading_error_previous
0,2023,COMEBACK_ROUTE_RIGHT,1628,5.667,5.597,1.199,4.045,1.417
1,2023,CURL_ROUTE_RIGHT,1454,5.344,5.367,1.425,3.748,1.765
2,2023,DAGGER_ROUTE_LEFT,2503,7.063,7.063,1.297,2.951,1.538
3,2023,GAUNTLET_DRILL,5761,7.674,7.723,0.924,2.541,0.961
4,2023,GO_ROUTE_RIGHT,2706,9.006,9.018,0.730,1.732,0.776
5,2023,OVER_SHOULDER_ADJUST,2137,6.368,6.480,0.917,2.244,0.946
6,2023,POST_CORNER_ROUTE_RIGHT,2256,7.782,7.778,1.002,2.418,1.210
7,2023,RED_ZONE_FADE_RIGHT,1272,5.270,5.283,1.053,2.715,1.188
8,2023,SLANT_ROUTE_LEFT,1360,6.872,6.937,1.111,2.823,1.652
9,2023,SLOT_SAIL_ROUTE_RIGHT,2331,7.454,7.427,1.420,3.083,1.657



4. RAW TRACKING COMPLETENESS


,draft_year,tracking_rows,players,missing_speed,missing_direction,missing_coordinates,median_speed
0,2023,30661,39,0,0,0,6.583
1,2024,24799,31,0,0,0,6.700
2,2025,28604,37,0,0,0,6.710



TRACKING CONSISTENCY AUDIT SAVED.


In [43]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

REPORTS.mkdir(parents=True, exist_ok=True)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

keys = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

base = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

turn = pd.read_parquet(
    FEATURES / "wr_combine_turning_candidate_features.parquet"
)

base = base[base["drill_name"].isin(PRIMARY_DRILLS)].copy()
turn = turn[turn["drill_name"].isin(PRIMARY_DRILLS)].copy()

# --------------------------------------------------
# 1. Confirm unique attempt keys
# --------------------------------------------------

print("BASE ROWS:", len(base))
print("TURNING ROWS:", len(turn))
print("BASE UNIQUE KEYS:", base[keys].drop_duplicates().shape[0])
print("TURNING UNIQUE KEYS:", turn[keys].drop_duplicates().shape[0])

assert not base.duplicated(keys).any()
assert not turn.duplicated(keys + ["speed_threshold"]).any()

# Attach original recorded turning and other attempt metrics
base_columns = [
    "recorded_total_turn",
    "recorded_mean_turn",
    "mean_speed",
    "peak_speed",
    "coordinate_path_length",
    "net_displacement",
    "path_efficiency",
    "observations",
    "duration_seconds"
]

base_columns = [c for c in base_columns if c in base.columns]

combined = turn.merge(
    base[keys + base_columns],
    on=keys,
    how="left",
    validate="many_to_one"
)

print("\nMERGED ROWS:", len(combined))
print("MISSING RECORDED TURN:",
      combined["recorded_total_turn"].isna().sum()
      if "recorded_total_turn" in combined.columns else "Not available")

# --------------------------------------------------
# 2. Compare turning across cohorts at thresholds 1 and 2
# --------------------------------------------------

metrics = [
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "turning_rate_degrees_per_second",
    "valid_pair_coverage"
]

threshold_summary = (
    combined.groupby(
        ["draft_year", "drill_name", "speed_threshold"]
    )[metrics]
    .median()
    .reset_index()
)

print("\n1. TURNING METRICS BY COHORT, DRILL AND THRESHOLD")
display(threshold_summary.round(3))

# --------------------------------------------------
# 3. Compare recorded and coordinate turning
# --------------------------------------------------

if "recorded_total_turn" in combined.columns:

    combined["coordinate_recorded_turn_ratio"] = (
        combined["coordinate_total_turn"] /
        combined["recorded_total_turn"].replace(0, np.nan)
    )

    agreement = (
        combined[combined["speed_threshold"] == 1.0]
        .groupby(["draft_year", "drill_name"])
        .agg(
            attempts=("nfl_id", "size"),
            coordinate_turn=("coordinate_total_turn", "median"),
            recorded_turn=("recorded_total_turn", "median"),
            turn_ratio=("coordinate_recorded_turn_ratio", "median")
        )
        .reset_index()
    )

    print("\n2. COORDINATE VS RECORDED TURNING")
    display(agreement.round(3))

    agreement.to_csv(
        REPORTS / "coordinate_vs_recorded_turn_by_cohort_drill.csv",
        index=False
    )

# --------------------------------------------------
# 4. Player-level drill medians
# --------------------------------------------------

# First summarize repeated attempts for each player and drill.
player_drill = (
    combined[combined["speed_threshold"] == 1.0]
    .groupby(["draft_year", "nfl_id", "drill_name"])
    .agg(
        coordinate_total_turn=("coordinate_total_turn", "median"),
        coordinate_mean_turn=("coordinate_mean_turn", "median"),
        coordinate_p95_turn=("coordinate_p95_turn", "median"),
        turning_rate=("turning_rate_degrees_per_second", "median"),
        valid_pair_coverage=("valid_pair_coverage", "median")
    )
    .reset_index()
)

player_distribution = (
    player_drill.groupby(["draft_year", "drill_name"])
    .agg(
        players=("nfl_id", "nunique"),
        median_total_turn=("coordinate_total_turn", "median"),
        p25_total_turn=("coordinate_total_turn",
                        lambda x: x.quantile(0.25)),
        p75_total_turn=("coordinate_total_turn",
                        lambda x: x.quantile(0.75)),
        median_mean_turn=("coordinate_mean_turn", "median"),
        median_p95_turn=("coordinate_p95_turn", "median"),
        median_turning_rate=("turning_rate", "median"),
        median_coverage=("valid_pair_coverage", "median")
    )
    .reset_index()
)

print("\n3. PLAYER-LEVEL TURNING DISTRIBUTIONS")
display(player_distribution.round(3))

# --------------------------------------------------
# 5. Cohort ratios relative to 2023
# --------------------------------------------------

turn_pivot = player_distribution.pivot(
    index="drill_name",
    columns="draft_year",
    values="median_total_turn"
)

turn_pivot["2024_vs_2023_pct"] = (
    (turn_pivot[2024] / turn_pivot[2023]) - 1
) * 100

turn_pivot["2025_vs_2023_pct"] = (
    (turn_pivot[2025] / turn_pivot[2023]) - 1
) * 100

print("\n4. PLAYER-LEVEL COHORT TURNING DIFFERENCES (%)")
display(turn_pivot.round(2))

# --------------------------------------------------
# 6. Save outputs
# --------------------------------------------------

threshold_summary.to_csv(
    REPORTS / "turning_threshold_sensitivity_by_cohort_drill.csv",
    index=False
)

player_drill.to_csv(
    REPORTS / "player_level_turning_by_drill.csv",
    index=False
)

player_distribution.to_csv(
    REPORTS / "player_turning_distribution_by_cohort_drill.csv",
    index=False
)

turn_pivot.to_csv(
    REPORTS / "player_turning_cohort_comparison.csv"
)

print("\nTURNING CONSISTENCY INVESTIGATION SAVED.")


BASE ROWS: 1282
TURNING ROWS: 2564
BASE UNIQUE KEYS: 1282
TURNING UNIQUE KEYS: 1282

MERGED ROWS: 2564
MISSING RECORDED TURN: Not available

1. TURNING METRICS BY COHORT, DRILL AND THRESHOLD


,draft_year,drill_name,speed_threshold,coordinate_total_turn,coordinate_mean_turn,coordinate_p95_turn,turning_rate_degrees_per_second,valid_pair_coverage
0,2023,COMEBACK_ROUTE_RIGHT,1.0,200.617,4.026,13.472,38.580,0.945
1,2023,COMEBACK_ROUTE_RIGHT,2.0,180.469,3.827,13.113,33.712,0.896
2,2023,CURL_ROUTE_RIGHT,1.0,303.750,5.901,17.883,56.421,0.963
3,2023,CURL_ROUTE_RIGHT,2.0,213.778,4.776,13.085,38.243,0.833
4,2023,DAGGER_ROUTE_LEFT,1.0,196.918,3.257,7.879,31.286,0.962
...,...,...,...,...,...,...,...,...
61,2025,SLANT_ROUTE_LEFT,2.0,64.745,2.862,4.709,24.311,0.846
62,2025,SLOT_SAIL_ROUTE_RIGHT,1.0,190.986,2.958,6.246,28.190,0.955
63,2025,SLOT_SAIL_ROUTE_RIGHT,2.0,188.683,3.013,6.265,27.777,0.923
64,2025,SLOT_STRIKE_ROUTE_LEFT,1.0,192.898,3.703,8.213,31.922,0.862



3. PLAYER-LEVEL TURNING DISTRIBUTIONS


,draft_year,drill_name,players,median_total_turn,p25_total_turn,p75_total_turn,median_mean_turn,median_p95_turn,median_turning_rate,median_coverage
0,2023,COMEBACK_ROUTE_RIGHT,36,199.602,177.123,215.301,4.019,13.511,38.602,0.945
1,2023,CURL_ROUTE_RIGHT,37,303.750,265.148,365.817,5.894,17.883,56.421,0.963
2,2023,DAGGER_ROUTE_LEFT,39,201.809,186.463,214.676,3.278,7.903,31.506,0.967
3,2023,GAUNTLET_DRILL,39,198.840,191.679,214.905,2.378,7.929,22.782,0.954
4,2023,GO_ROUTE_RIGHT,35,118.659,106.552,127.506,1.701,4.764,16.029,0.958
5,2023,OVER_SHOULDER_ADJUST,37,129.818,109.018,136.389,2.106,6.340,20.664,0.982
6,2023,POST_CORNER_ROUTE_RIGHT,36,157.496,147.201,171.681,2.543,6.322,24.700,0.969
7,2023,RED_ZONE_FADE_RIGHT,36,138.191,117.036,158.402,3.383,11.598,29.255,0.890
8,2023,SLANT_ROUTE_LEFT,39,123.660,110.179,141.176,3.232,8.033,27.046,0.844
9,2023,SLOT_SAIL_ROUTE_RIGHT,33,237.236,205.236,250.744,3.488,7.990,30.900,0.889



4. PLAYER-LEVEL COHORT TURNING DIFFERENCES (%)


draft_year,2023,2024,2025,2024_vs_2023_pct,2025_vs_2023_pct
drill_name,,,,,
COMEBACK_ROUTE_RIGHT,199.60,201.02,188.99,0.71,-5.32
CURL_ROUTE_RIGHT,303.75,318.76,261.75,4.94,-13.83
DAGGER_ROUTE_LEFT,201.81,198.87,148.58,-1.46,-26.38
GAUNTLET_DRILL,198.84,189.05,117.10,-4.92,-41.11
GO_ROUTE_RIGHT,118.66,118.95,40.87,0.24,-65.56
OVER_SHOULDER_ADJUST,129.82,108.97,40.06,-16.06,-69.14
POST_CORNER_ROUTE_RIGHT,157.50,182.57,88.30,15.92,-43.94
RED_ZONE_FADE_RIGHT,138.19,145.00,62.14,4.93,-55.03
SLANT_ROUTE_LEFT,123.66,125.45,70.36,1.45,-43.10



TURNING CONSISTENCY INVESTIGATION SAVED.


In [44]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

base = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

turn = pd.read_parquet(
    FEATURES / "wr_combine_turning_candidate_features.parquet"
)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

keys = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

# --------------------------------------------------
# 1. Inspect available recorded-turn fields
# --------------------------------------------------

print("1. TURN-RELATED COLUMNS IN BASE DATA")
print([
    c for c in base.columns
    if "turn" in c.lower() or "direction" in c.lower()
])

print("\nTURNING CANDIDATE COLUMNS")
print([
    c for c in turn.columns
    if "turn" in c.lower() or "direction" in c.lower()
])

# --------------------------------------------------
# 2. Threshold sensitivity by cohort and drill
# --------------------------------------------------

t = turn[
    turn["drill_name"].isin(PRIMARY_DRILLS)
].copy()

thresholds = (
    t.groupby(
        ["draft_year", "drill_name", "speed_threshold"]
    )
    .agg(
        attempts=("nfl_id", "size"),
        players=("nfl_id", "nunique"),
        total_turn=("coordinate_total_turn", "median"),
        mean_turn=("coordinate_mean_turn", "median"),
        p95_turn=("coordinate_p95_turn", "median"),
        turning_rate=("turning_rate_degrees_per_second", "median"),
        coverage=("valid_pair_coverage", "median")
    )
    .reset_index()
)

wide = thresholds.pivot(
    index=["draft_year", "drill_name"],
    columns="speed_threshold",
    values=["total_turn", "mean_turn", "p95_turn",
            "turning_rate", "coverage"]
)

wide.columns = [
    f"{metric}_threshold_{threshold}"
    for metric, threshold in wide.columns
]

wide = wide.reset_index()

wide["total_turn_threshold_change_pct"] = (
    wide["total_turn_threshold_2.0"] /
    wide["total_turn_threshold_1.0"] - 1
) * 100

print("\n2. THRESHOLD 1 VS 2")
display(wide.round(3))

# --------------------------------------------------
# 3. Threshold agreement at player level
# --------------------------------------------------

player_threshold = (
    t.groupby(
        ["draft_year", "nfl_id", "drill_name", "speed_threshold"]
    )["coordinate_total_turn"]
    .median()
    .reset_index()
)

player_wide = player_threshold.pivot(
    index=["draft_year", "nfl_id", "drill_name"],
    columns="speed_threshold",
    values="coordinate_total_turn"
).reset_index()

print("\n3. PLAYER-LEVEL THRESHOLD AGREEMENT")

for year, group in player_wide.groupby("draft_year"):
    valid = group.dropna(subset=[1.0, 2.0])

    print(
        year,
        "| paired players/drills:", len(valid),
        "| Spearman:",
        round(valid[1.0].corr(valid[2.0], method="spearman"), 3),
        "| Pearson:",
        round(valid[1.0].corr(valid[2.0]), 3)
    )

# --------------------------------------------------
# 4. Look for recorded turning in source files
# --------------------------------------------------

recorded_candidates = [
    c for c in base.columns
    if "recorded" in c.lower()
    or "direction" in c.lower()
]

print("\n4. POSSIBLE RECORDED-TURN FIELDS")
print(recorded_candidates)

if "recorded_total_turn" in base.columns:
    print("\nRECORDED TURNING IS AVAILABLE.")
    print(
        base.groupby(["draft_year", "drill_name"])
        ["recorded_total_turn"]
        .median()
        .round(3)
    )
else:
    print(
        "\nRECORDED TOTAL TURN IS NOT IN THIS BASE DATASET. "
        "Do not substitute another field without checking its definition."
    )

# --------------------------------------------------
# 5. Save compact sensitivity outputs
# --------------------------------------------------

wide.to_csv(
    REPORTS / "threshold_sensitivity_compact_by_drill.csv",
    index=False
)

player_wide.to_csv(
    REPORTS / "player_threshold_agreement.csv",
    index=False
)

print("\nSENSITIVITY OUTPUTS SAVED.")


1. TURN-RELATED COLUMNS IN BASE DATA
['total_direction_change', 'valid_turn_pairs', 'coordinate_total_turn', 'coordinate_mean_turn', 'coordinate_median_turn', 'coordinate_p95_turn', 'coordinate_max_turn', 'recorded_total_turn', 'recorded_mean_turn', 'turning_rate_degrees_per_second', 'turning_coverage_flag']

TURNING CANDIDATE COLUMNS
['valid_turn_pairs', 'coordinate_total_turn', 'coordinate_mean_turn', 'coordinate_median_turn', 'coordinate_p95_turn', 'coordinate_max_turn', 'recorded_total_turn', 'recorded_mean_turn', 'turning_rate_degrees_per_second']

2. THRESHOLD 1 VS 2


,draft_year,drill_name,total_turn_threshold_1.0,total_turn_threshold_2.0,mean_turn_threshold_1.0,mean_turn_threshold_2.0,p95_turn_threshold_1.0,p95_turn_threshold_2.0,turning_rate_threshold_1.0,turning_rate_threshold_2.0,coverage_threshold_1.0,coverage_threshold_2.0,total_turn_threshold_change_pct
0,2023,COMEBACK_ROUTE_RIGHT,200.617,180.469,4.026,3.827,13.472,13.113,38.580,33.712,0.945,0.896,-10.043
1,2023,CURL_ROUTE_RIGHT,303.750,213.778,5.901,4.776,17.883,13.085,56.421,38.243,0.963,0.833,-29.620
2,2023,DAGGER_ROUTE_LEFT,196.918,181.133,3.257,3.079,7.879,7.144,31.286,28.070,0.962,0.922,-8.016
3,2023,GAUNTLET_DRILL,199.298,179.177,2.350,2.182,7.993,6.941,22.441,20.245,0.955,0.926,-10.096
4,2023,GO_ROUTE_RIGHT,117.211,93.790,1.701,1.458,4.764,4.073,16.029,13.214,0.958,0.920,-19.982
5,2023,OVER_SHOULDER_ADJUST,130.467,108.245,2.126,1.860,6.340,4.907,20.861,17.283,0.982,0.934,-17.032
6,2023,POST_CORNER_ROUTE_RIGHT,159.201,142.297,2.548,2.373,6.334,5.711,24.707,22.167,0.969,0.936,-10.618
7,2023,RED_ZONE_FADE_RIGHT,140.856,85.643,3.474,2.543,11.484,6.258,30.152,18.567,0.886,0.744,-39.198
8,2023,SLANT_ROUTE_LEFT,123.660,104.235,3.232,2.978,8.033,6.932,27.046,23.637,0.844,0.800,-15.708
9,2023,SLOT_SAIL_ROUTE_RIGHT,229.179,211.669,3.416,3.315,7.979,7.401,30.213,28.690,0.888,0.855,-7.640



3. PLAYER-LEVEL THRESHOLD AGREEMENT
2023 | paired players/drills: 404 | Spearman: 0.864 | Pearson: 0.797
2024 | paired players/drills: 325 | Spearman: 0.915 | Pearson: 0.883
2025 | paired players/drills: 384 | Spearman: 0.934 | Pearson: 0.891

4. POSSIBLE RECORDED-TURN FIELDS
['summed_recorded_displacement', 'total_direction_change', 'recorded_total_turn', 'recorded_mean_turn']

RECORDED TURNING IS AVAILABLE.
draft_year  drill_name             
2023        COMEBACK_ROUTE_RIGHT       137.257
            CURL_ROUTE_RIGHT           242.854
            DAGGER_ROUTE_LEFT          148.428
            FORTY_YARD_DASH             13.538
            GAUNTLET_DRILL             115.119
            GO_ROUTE_RIGHT              39.575
            OVER_SHOULDER_ADJUST        25.222
            POST_CORNER_ROUTE_LEFT     133.194
            POST_CORNER_ROUTE_RIGHT    109.062
            RED_ZONE_FADE_RIGHT         63.146
            SHORT_SHUTTLE               70.790
            SLANT_ROUTE_LEFT     

In [45]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
REPORTS = ROOT / "reports" / "data_audit"

base = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

turn = pd.read_parquet(
    FEATURES / "wr_combine_turning_candidate_features.parquet"
)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

keys = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

# --------------------------------------------------
# 1. Select primary drills and threshold 1
# --------------------------------------------------

b = base[base["drill_name"].isin(PRIMARY_DRILLS)].copy()

t = turn[
    (turn["drill_name"].isin(PRIMARY_DRILLS)) &
    (turn["speed_threshold"] == 1.0)
].copy()

assert not b.duplicated(keys).any()
assert not t.duplicated(keys).any()

# Merge recorded and coordinate turning at attempt level
paired = b[
    keys + [
        "recorded_total_turn",
        "recorded_mean_turn",
        "total_direction_change"
    ]
].merge(
    t[
        keys + [
            "coordinate_total_turn",
            "coordinate_mean_turn",
            "coordinate_p95_turn",
            "turning_rate_degrees_per_second",
            "valid_pair_coverage"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

print("PAIRED ATTEMPTS:", len(paired))
print("PAIRED PLAYERS:", paired["nfl_id"].nunique())
print("MISSING RECORDED TURN:",
      paired["recorded_total_turn"].isna().sum())
print("MISSING COORDINATE TURN:",
      paired["coordinate_total_turn"].isna().sum())

# --------------------------------------------------
# 2. Check whether recorded fields are equivalent
# --------------------------------------------------

paired["recorded_field_difference"] = (
    paired["total_direction_change"] -
    paired["recorded_total_turn"]
)

print("\n1. total_direction_change VS recorded_total_turn")
display(
    paired["recorded_field_difference"]
    .describe()
    .to_frame("difference")
    .round(5)
)

print(
    "Exact equality:",
    np.allclose(
        paired["total_direction_change"],
        paired["recorded_total_turn"],
        equal_nan=True
    )
)

# --------------------------------------------------
# 3. Aggregate repeated attempts to player-drill
# --------------------------------------------------

player_drill = (
    paired.groupby(["draft_year", "nfl_id", "drill_name"])
    .agg(
        attempts=("event_id", "size"),
        recorded_turn=("recorded_total_turn", "median"),
        coordinate_turn=("coordinate_total_turn", "median"),
        recorded_mean_turn=("recorded_mean_turn", "median"),
        coordinate_mean_turn=("coordinate_mean_turn", "median"),
        coordinate_p95_turn=("coordinate_p95_turn", "median"),
        turning_rate=("turning_rate_degrees_per_second", "median"),
        coverage=("valid_pair_coverage", "median")
    )
    .reset_index()
)

# --------------------------------------------------
# 4. Compare measures within each cohort and drill
# --------------------------------------------------

comparison = (
    player_drill.groupby(["draft_year", "drill_name"])
    .agg(
        players=("nfl_id", "nunique"),
        recorded_turn_median=("recorded_turn", "median"),
        coordinate_turn_median=("coordinate_turn", "median"),
        recorded_turn_p25=("recorded_turn",
                           lambda x: x.quantile(.25)),
        recorded_turn_p75=("recorded_turn",
                           lambda x: x.quantile(.75)),
        coordinate_turn_p25=("coordinate_turn",
                             lambda x: x.quantile(.25)),
        coordinate_turn_p75=("coordinate_turn",
                             lambda x: x.quantile(.75))
    )
    .reset_index()
)

comparison["coordinate_recorded_ratio"] = (
    comparison["coordinate_turn_median"] /
    comparison["recorded_turn_median"].replace(0, np.nan)
)

print("\n2. PLAYER-LEVEL RECORDED VS COORDINATE TURNING")
display(comparison.round(3))

# --------------------------------------------------
# 5. Cohort changes relative to 2023
# --------------------------------------------------

for metric in ["recorded_turn_median", "coordinate_turn_median"]:
    pivot = comparison.pivot(
        index="drill_name",
        columns="draft_year",
        values=metric
    )

    pivot[f"{metric}_2025_vs_2023_pct"] = (
        (pivot[2025] / pivot[2023]) - 1
    ) * 100

    if metric == "recorded_turn_median":
        recorded_change = pivot[
            f"{metric}_2025_vs_2023_pct"
        ].rename("recorded_2025_vs_2023_pct")
    else:
        coordinate_change = pivot[
            f"{metric}_2025_vs_2023_pct"
        ].rename("coordinate_2025_vs_2023_pct")

cohort_change = pd.concat(
    [recorded_change, coordinate_change],
    axis=1
)

print("\n3. 2025 CHANGE BY MEASUREMENT METHOD")
display(cohort_change.round(2))

# --------------------------------------------------
# 6. Correlation between measures by cohort
# --------------------------------------------------

correlations = []

for year, group in player_drill.groupby("draft_year"):

    correlations.append({
        "draft_year": year,
        "player_drill_pairs": len(group),
        "spearman": group["recorded_turn"].corr(
            group["coordinate_turn"], method="spearman"
        ),
        "pearson": group["recorded_turn"].corr(
            group["coordinate_turn"], method="pearson"
        )
    })

correlations = pd.DataFrame(correlations)

print("\n4. RECORDED VS COORDINATE CORRELATION")
display(correlations.round(3))

# --------------------------------------------------
# 7. Save
# --------------------------------------------------

comparison.to_csv(
    REPORTS / "paired_recorded_coordinate_turn_by_drill.csv",
    index=False
)

cohort_change.to_csv(
    REPORTS / "recorded_coordinate_turn_cohort_changes.csv"
)

correlations.to_csv(
    REPORTS / "recorded_coordinate_turn_correlations.csv",
    index=False
)

player_drill.to_parquet(
    FEATURES / "wr_combine_player_drill_paired_turning.parquet",
    index=False
)

print("\nPAIRED MEASUREMENT AUDIT SAVED.")


PAIRED ATTEMPTS: 1282
PAIRED PLAYERS: 107
MISSING RECORDED TURN: 0
MISSING COORDINATE TURN: 0

1. total_direction_change VS recorded_total_turn


,difference
count,1282.00000
mean,58.61314
std,88.43769
min,0.01703
25%,3.64325
50%,12.20686
75%,100.82210
max,1436.81257


Exact equality: False

2. PLAYER-LEVEL RECORDED VS COORDINATE TURNING


,draft_year,drill_name,players,recorded_turn_median,coordinate_turn_median,recorded_turn_p25,recorded_turn_p75,coordinate_turn_p25,coordinate_turn_p75,coordinate_recorded_ratio
0,2023,COMEBACK_ROUTE_RIGHT,36,136.746,199.602,130.389,157.482,177.123,215.301,1.460
1,2023,CURL_ROUTE_RIGHT,37,235.660,303.750,194.062,292.073,265.148,365.817,1.289
2,2023,DAGGER_ROUTE_LEFT,39,148.428,201.809,136.030,158.467,186.463,214.676,1.360
3,2023,GAUNTLET_DRILL,39,115.940,198.840,109.513,120.229,191.679,214.905,1.715
4,2023,GO_ROUTE_RIGHT,35,40.317,118.659,28.141,49.114,106.552,127.506,2.943
5,2023,OVER_SHOULDER_ADJUST,37,25.673,129.818,20.863,33.254,109.018,136.389,5.057
6,2023,POST_CORNER_ROUTE_RIGHT,36,109.062,157.496,99.772,119.127,147.201,171.681,1.444
7,2023,RED_ZONE_FADE_RIGHT,36,63.004,138.191,45.155,75.550,117.036,158.402,2.193
8,2023,SLANT_ROUTE_LEFT,39,93.170,123.660,82.683,105.039,110.179,141.176,1.327
9,2023,SLOT_SAIL_ROUTE_RIGHT,33,196.530,237.236,164.841,203.013,205.236,250.744,1.207



3. 2025 CHANGE BY MEASUREMENT METHOD


,recorded_2025_vs_2023_pct,coordinate_2025_vs_2023_pct
drill_name,,
COMEBACK_ROUTE_RIGHT,33.36,-5.32
CURL_ROUTE_RIGHT,7.73,-13.83
DAGGER_ROUTE_LEFT,-4.77,-26.38
GAUNTLET_DRILL,-2.14,-41.11
GO_ROUTE_RIGHT,-21.71,-65.56
OVER_SHOULDER_ADJUST,31.13,-69.14
POST_CORNER_ROUTE_RIGHT,-25.57,-43.94
RED_ZONE_FADE_RIGHT,-8.60,-55.03
SLANT_ROUTE_LEFT,-30.50,-43.10



4. RECORDED VS COORDINATE CORRELATION


,draft_year,player_drill_pairs,spearman,pearson
0,2023,404,0.878,0.892
1,2024,325,0.919,0.927
2,2025,384,0.996,0.997



PAIRED MEASUREMENT AUDIT SAVED.


In [46]:

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURES = ROOT / "data" / "features"
AUDIT = ROOT / "reports" / "data_audit"
AUDIT.mkdir(parents=True, exist_ok=True)

KEYS = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

base = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

turning = pd.read_parquet(
    FEATURES / "wr_combine_turning_candidate_features.parquet"
)

# Keep only the primary drills and threshold 1
PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

base = base[base["drill_name"].isin(PRIMARY_DRILLS)].copy()

turn1 = turning[
    (turning["drill_name"].isin(PRIMARY_DRILLS)) &
    (turning["speed_threshold"] == 1.0)
].copy()

# Verify uniqueness before joining
assert not base.duplicated(KEYS).any(), "Duplicate base attempt keys"
assert not turn1.duplicated(KEYS).any(), "Duplicate threshold-1 keys"

# Rename overlapping turning columns
turn1 = turn1[
    KEYS + [
        "coordinate_total_turn",
        "coordinate_mean_turn",
        "coordinate_p95_turn",
        "recorded_total_turn",
        "recorded_mean_turn",
        "turning_rate_degrees_per_second",
        "valid_pair_coverage"
    ]
].copy()

turn1 = turn1.rename(columns={
    "coordinate_total_turn": "coordinate_total_turn_t1",
    "coordinate_mean_turn": "coordinate_mean_turn_t1",
    "coordinate_p95_turn": "coordinate_p95_turn_t1",
    "recorded_total_turn": "recorded_total_turn_t1",
    "recorded_mean_turn": "recorded_mean_turn_t1"
})

df = base.merge(
    turn1,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

print("PAIRED ATTEMPTS:", len(df))
print("UNIQUE PLAYERS:", df["nfl_id"].nunique())

# ---------------------------------------------------------
# 1. Compare the three existing turning measures
# ---------------------------------------------------------
comparison_cols = [
    "total_direction_change",
    "recorded_total_turn_t1",
    "coordinate_total_turn_t1"
]

print("\nMEASUREMENT SUMMARY")
display(df[comparison_cols].describe().round(3))

print("\nCORRELATIONS")
display(df[comparison_cols].corr(method="spearman").round(3))

# ---------------------------------------------------------
# 2. Difference and agreement diagnostics
# ---------------------------------------------------------
pairs = [
    ("total_direction_change", "recorded_total_turn_t1",
     "base_vs_recorded"),
    ("total_direction_change", "coordinate_total_turn_t1",
     "base_vs_coordinate"),
    ("recorded_total_turn_t1", "coordinate_total_turn_t1",
     "recorded_vs_coordinate")
]

agreement_rows = []

for col_a, col_b, label in pairs:
    valid = df[[col_a, col_b]].dropna()
    difference = valid[col_a] - valid[col_b]

    agreement_rows.append({
        "comparison": label,
        "n": len(valid),
        "pearson": valid[col_a].corr(valid[col_b]),
        "spearman": valid[col_a].corr(valid[col_b], method="spearman"),
        "mean_difference": difference.mean(),
        "median_absolute_difference": difference.abs().median(),
        "mean_absolute_difference": difference.abs().mean(),
        "p95_absolute_difference": difference.abs().quantile(.95)
    })

agreement = pd.DataFrame(agreement_rows)
display(agreement.round(3))

# ---------------------------------------------------------
# 3. Cohort-specific agreement
# ---------------------------------------------------------
cohort_rows = []

for year, group in df.groupby("draft_year"):
    for col_a, col_b, label in pairs:
        valid = group[[col_a, col_b]].dropna()
        difference = valid[col_a] - valid[col_b]

        cohort_rows.append({
            "draft_year": year,
            "comparison": label,
            "n": len(valid),
            "spearman": valid[col_a].corr(
                valid[col_b], method="spearman"
            ),
            "pearson": valid[col_a].corr(valid[col_b]),
            "median_absolute_difference": difference.abs().median(),
            "mean_absolute_difference": difference.abs().mean(),
            "p95_absolute_difference": difference.abs().quantile(.95)
        })

cohort_agreement = pd.DataFrame(cohort_rows)
display(cohort_agreement.round(3))

# ---------------------------------------------------------
# 4. Identify the largest disagreements
# ---------------------------------------------------------
df["base_recorded_abs_diff"] = (
    df["total_direction_change"] -
    df["recorded_total_turn_t1"]
).abs()

df["base_coordinate_abs_diff"] = (
    df["total_direction_change"] -
    df["coordinate_total_turn_t1"]
).abs()

df["recorded_coordinate_abs_diff"] = (
    df["recorded_total_turn_t1"] -
    df["coordinate_total_turn_t1"]
).abs()

print("\nLARGEST BASE VS RECORDED DISAGREEMENTS")
display(
    df.sort_values("base_recorded_abs_diff", ascending=False)[
        KEYS + [
            "total_direction_change",
            "recorded_total_turn_t1",
            "coordinate_total_turn_t1",
            "base_recorded_abs_diff"
        ]
    ].head(15)
)

# ---------------------------------------------------------
# 5. Drill-level differences by cohort
# ---------------------------------------------------------
drill_differences = (
    df.groupby(["draft_year", "drill_name"])
    .agg(
        attempts=("nfl_id", "size"),
        players=("nfl_id", "nunique"),
        base_turn_median=("total_direction_change", "median"),
        recorded_turn_median=("recorded_total_turn_t1", "median"),
        coordinate_turn_median=("coordinate_total_turn_t1", "median"),
        base_recorded_abs_diff=("base_recorded_abs_diff", "median"),
        base_coordinate_abs_diff=("base_coordinate_abs_diff", "median"),
        recorded_coordinate_abs_diff=("recorded_coordinate_abs_diff", "median")
    )
    .reset_index()
)

display(drill_differences.round(3))

# ---------------------------------------------------------
# 6. Save all diagnostics
# ---------------------------------------------------------
agreement.to_csv(
    AUDIT / "turning_measurement_agreement_overall.csv",
    index=False
)

cohort_agreement.to_csv(
    AUDIT / "turning_measurement_agreement_by_cohort.csv",
    index=False
)

drill_differences.to_csv(
    AUDIT / "turning_measurement_agreement_by_drill.csv",
    index=False
)

df.to_parquet(
    FEATURES / "wr_turning_measurement_comparison.parquet",
    index=False
)

print("\nTURNING MEASUREMENT COMPARISON SAVED")


PAIRED ATTEMPTS: 1282
UNIQUE PLAYERS: 107

MEASUREMENT SUMMARY


,total_direction_change,recorded_total_turn_t1,coordinate_total_turn_t1
count,1282.000,1282.000,1282.000
mean,182.775,124.162,169.650
std,120.278,70.804,76.294
min,13.832,10.689,18.888
25%,90.247,69.988,113.933
50%,154.232,118.087,168.174
75%,266.923,164.980,211.312
max,1613.416,376.919,525.236



CORRELATIONS


,total_direction_change,recorded_total_turn_t1,coordinate_total_turn_t1
total_direction_change,1.000,0.793,0.675
recorded_total_turn_t1,0.793,1.000,0.854
coordinate_total_turn_t1,0.675,0.854,1.000


,comparison,n,pearson,spearman,mean_difference,median_absolute_difference,mean_absolute_difference,p95_absolute_difference
0,base_vs_recorded,1282,0.685,0.793,58.613,12.207,58.613,222.530
1,base_vs_coordinate,1282,0.603,0.675,13.125,51.178,66.331,180.828
2,recorded_vs_coordinate,1282,0.872,0.854,-45.488,42.819,45.497,112.521


,draft_year,comparison,n,spearman,pearson,median_absolute_difference,mean_absolute_difference,p95_absolute_difference
0,2023,base_vs_recorded,468,0.841,0.778,14.410,55.770,212.384
1,2023,base_vs_coordinate,468,0.704,0.659,62.792,74.260,172.387
2,2023,recorded_vs_coordinate,468,0.867,0.885,60.841,65.697,119.088
3,2024,base_vs_recorded,377,0.685,0.529,13.315,62.139,223.142
4,2024,base_vs_coordinate,377,0.702,0.544,54.988,68.237,156.193
5,2024,recorded_vs_coordinate,377,0.896,0.919,61.658,64.525,116.103
6,2025,base_vs_recorded,437,0.802,0.725,9.167,58.616,228.840
7,2025,base_vs_coordinate,437,0.798,0.726,9.628,56.196,224.348
8,2025,recorded_vs_coordinate,437,0.996,0.997,6.221,7.449,17.359



LARGEST BASE VS RECORDED DISAGREEMENTS


,draft_year,nfl_id,drill_type,drill_name,attempt,event_id,total_direction_change,recorded_total_turn_t1,coordinate_total_turn_t1,base_recorded_abs_diff
568,2024,57606,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,1,20240302100700_1709414917288_000,1613.415985,176.603411,275.104819,1436.812574
1113,2023,55887,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,1,6414e5fde3d8e72dcdc8d3a0-2023,531.623943,158.423450,189.854851,373.200493
507,2025,58493,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,10,20250301100700_1740866498334_000,413.425932,42.824498,55.425015,370.601434
436,2023,55965,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,1,6414e684e3d8e72dcdcb5003-2023,443.826089,76.000295,218.224826,367.825794
107,2024,57152,SKILL_DRILLS_WR,GAUNTLET_DRILL,2,20240302100800_1709422428077_000,480.362571,116.838998,184.596917,363.523573
696,2023,55998,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,1,6414e684e3d8e72dcdcb4ed2-2023,517.854147,169.200999,195.117588,348.653148
156,2023,55928,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,1,6414e684e3d8e72dcdcb4ebe-2023,512.450222,170.700902,212.423295,341.749320
103,2024,57130,SKILL_DRILLS_WR,GAUNTLET_DRILL,2,20240302100800_1709422416995_000,430.313705,95.461512,166.352298,334.852193
773,2023,56060,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,1,6414e684e3d8e72dcdcb4eca-2023,494.840638,166.688272,271.439831,328.152366
1191,2025,58751,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,13,20250301100700_1740866834858_000,532.208701,205.347116,212.165731,326.861586


,draft_year,drill_name,attempts,players,base_turn_median,recorded_turn_median,coordinate_turn_median,base_recorded_abs_diff,base_coordinate_abs_diff,recorded_coordinate_abs_diff
0,2023,COMEBACK_ROUTE_RIGHT,37,36,151.424,137.257,200.617,7.892,39.851,57.118
1,2023,CURL_ROUTE_RIGHT,43,37,268.259,242.854,303.750,4.949,50.756,58.879
2,2023,DAGGER_ROUTE_LEFT,43,39,155.324,148.428,196.918,7.478,47.573,53.319
3,2023,GAUNTLET_DRILL,71,39,147.526,115.119,199.298,27.353,49.828,86.093
4,2023,GO_ROUTE_RIGHT,41,35,48.029,39.575,117.211,8.774,73.240,75.517
5,2023,OVER_SHOULDER_ADJUST,38,37,26.486,25.222,130.467,1.864,98.514,101.070
6,2023,POST_CORNER_ROUTE_RIGHT,38,36,113.601,109.062,159.201,4.052,46.694,50.738
7,2023,RED_ZONE_FADE_RIGHT,43,36,79.224,63.146,140.856,16.953,72.741,76.383
8,2023,SLANT_ROUTE_LEFT,39,39,233.473,93.170,123.660,130.390,90.176,29.324
9,2023,SLOT_SAIL_ROUTE_RIGHT,38,33,325.770,182.008,229.179,158.109,112.998,45.966



TURNING MEASUREMENT COMPARISON SAVED


In [48]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
RAW = ROOT / "data/raw/nfl-big-data-bowl-2027/combine_tracking.csv"
FEATURES = ROOT / "data/features"
AUDIT = ROOT / "reports/data_audit"
AUDIT.mkdir(parents=True, exist_ok=True)

base = pd.read_parquet(
    FEATURES / "wr_combine_attempt_features_enriched.parquet"
)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL", "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT", "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT", "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT", "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT", "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

KEYS = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

base = base[base["drill_name"].isin(PRIMARY_DRILLS)].copy()

raw = pd.read_csv(
    RAW,
    usecols=[
        "draft_year", "event_id", "nfl_id",
        "entity_type", "time", "drill_type",
        "drill_name", "attempt", "x", "y", "s", "dir"
    ]
)

raw = raw[
    (raw["entity_type"] == "PLAYER") &
    (raw["drill_name"].isin(PRIMARY_DRILLS)) &
    (raw["nfl_id"].isin(base["nfl_id"].unique()))
].copy()

raw["time"] = pd.to_datetime(raw["time"], errors="coerce")

for col in ["x", "y", "s", "dir", "attempt", "draft_year"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

raw = raw.dropna(subset=KEYS + ["time", "x", "y", "s", "dir"])
raw = raw.sort_values(KEYS + ["time"])

print("RAW PLAYER OBSERVATIONS:", len(raw))
print("RAW ATTEMPTS:", raw.groupby(KEYS).ngroups)

def circular_difference(current, previous):
    return ((current - previous + 180) % 360) - 180

results = []

for key, g in raw.groupby(KEYS, sort=False):

    g = g.sort_values("time")

    x = g["x"].to_numpy()
    y = g["y"].to_numpy()
    speed = g["s"].to_numpy()
    direction = g["dir"].to_numpy()

    # Direction changes from recorded direction
    recorded_diff = circular_difference(
        direction[1:], direction[:-1]
    )

    # Coordinate heading for each consecutive position pair
    dx = np.diff(x)
    dy = np.diff(y)

    heading = np.degrees(np.arctan2(dx, dy)) % 360

    # Coordinate direction change between successive headings
    coordinate_diff = circular_difference(
        heading[1:], heading[:-1]
    )

    # Align recorded direction changes to the same middle
    # observation used by coordinate turning.
    recorded_aligned = recorded_diff[1:]

    # Both adjacent movement segments must meet threshold.
    # This produces n-2 valid pairs, matching both turn arrays.
    middle_speed = speed[1:-1]
    valid = np.ones(len(coordinate_diff), dtype=bool)

    row = dict(zip(KEYS, key))

    row["recorded_turn_t0"] = np.abs(recorded_aligned).sum()
    row["coordinate_turn_t0"] = np.abs(coordinate_diff).sum()

    for threshold in [1.0, 2.0]:

        valid = (
            (speed[:-2] >= threshold) &
            (speed[1:-1] >= threshold) &
            (speed[2:] >= threshold)
        )

        row[f"recorded_turn_t{threshold}"] = (
            np.abs(recorded_aligned[valid]).sum()
        )

        row[f"coordinate_turn_t{threshold}"] = (
            np.abs(coordinate_diff[valid]).sum()
        )

        row[f"valid_pairs_t{threshold}"] = int(valid.sum())

        row[f"coverage_t{threshold}"] = (
            valid.sum() / max(len(valid), 1)
        )

    results.append(row)

reconstructed = pd.DataFrame(results)

print("\nRECONSTRUCTED ATTEMPTS:", len(reconstructed))

stored_cols = [
    "total_direction_change",
    "recorded_total_turn",
    "coordinate_total_turn"
]

comparison = base[KEYS + stored_cols].merge(
    reconstructed,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

print("MATCHED ATTEMPTS:", len(comparison))

comparisons = [
    ("recorded_total_turn", "recorded_turn_t0"),
    ("recorded_total_turn", "recorded_turn_t1.0"),
    ("recorded_total_turn", "recorded_turn_t2.0"),
    ("coordinate_total_turn", "coordinate_turn_t0"),
    ("coordinate_total_turn", "coordinate_turn_t1.0"),
    ("coordinate_total_turn", "coordinate_turn_t2.0"),
    ("total_direction_change", "recorded_turn_t0"),
    ("total_direction_change", "coordinate_turn_t0")
]

audit_rows = []

for stored, rebuilt in comparisons:

    valid_data = comparison[[stored, rebuilt]].dropna()
    difference = valid_data[stored] - valid_data[rebuilt]

    audit_rows.append({
        "stored_field": stored,
        "reconstructed_field": rebuilt,
        "n": len(valid_data),
        "pearson": valid_data[stored].corr(valid_data[rebuilt]),
        "spearman": valid_data[stored].corr(
            valid_data[rebuilt], method="spearman"
        ),
        "median_absolute_error": difference.abs().median(),
        "mean_absolute_error": difference.abs().mean(),
        "p95_absolute_error": difference.abs().quantile(.95)
    })

reconstruction_audit = pd.DataFrame(audit_rows)

print("\nRECONSTRUCTION VALIDATION")
display(reconstruction_audit.round(4))

cohort_rows = []

for year, group in comparison.groupby("draft_year"):

    for stored, rebuilt in comparisons:

        valid_data = group[[stored, rebuilt]].dropna()
        difference = valid_data[stored] - valid_data[rebuilt]

        cohort_rows.append({
            "draft_year": year,
            "stored_field": stored,
            "reconstructed_field": rebuilt,
            "n": len(valid_data),
            "median_absolute_error": difference.abs().median(),
            "mean_absolute_error": difference.abs().mean(),
            "p95_absolute_error": difference.abs().quantile(.95)
        })

cohort_audit = pd.DataFrame(cohort_rows)

print("\nCOHORT RECONSTRUCTION AUDIT")
display(cohort_audit.round(4))

reconstruction_audit.to_csv(
    AUDIT / "turning_reconstruction_overall.csv",
    index=False
)

cohort_audit.to_csv(
    AUDIT / "turning_reconstruction_by_cohort.csv",
    index=False
)

comparison.to_parquet(
    FEATURES / "wr_turning_reconstruction_validation.parquet",
    index=False
)

print("\nRECONSTRUCTION AUDIT SAVED.")


RAW PLAYER OBSERVATIONS: 84064
RAW ATTEMPTS: 1282

RECONSTRUCTED ATTEMPTS: 1282
MATCHED ATTEMPTS: 1282

RECONSTRUCTION VALIDATION


,stored_field,reconstructed_field,n,pearson,spearman,median_absolute_error,mean_absolute_error,p95_absolute_error
0,recorded_total_turn,recorded_turn_t0,1282,0.7145,0.8397,5.8863,47.1565,199.4016
1,recorded_total_turn,recorded_turn_t1.0,1282,0.9983,0.9987,1.4295,2.7222,11.3567
2,recorded_total_turn,recorded_turn_t2.0,1282,0.8626,0.9189,4.5408,19.7990,106.8455
3,coordinate_total_turn,coordinate_turn_t0,1282,0.4467,0.6780,29.1832,113.0455,469.5929
4,coordinate_total_turn,coordinate_turn_t1.0,1282,0.9935,0.9912,4.3987,7.9307,23.4668
5,coordinate_total_turn,coordinate_turn_t2.0,1282,0.8471,0.8766,22.6199,34.8773,128.3462
6,total_direction_change,recorded_turn_t0,1282,0.9810,0.9757,4.1566,11.4566,46.0181
7,total_direction_change,coordinate_turn_t0,1282,0.7571,0.8273,60.3066,102.7456,390.9129



COHORT RECONSTRUCTION AUDIT


,draft_year,stored_field,reconstructed_field,n,median_absolute_error,mean_absolute_error,p95_absolute_error
0,2023,recorded_total_turn,recorded_turn_t0,468,6.5994,45.8541,197.0950
1,2023,recorded_total_turn,recorded_turn_t1.0,468,1.4013,2.7153,11.2758
2,2023,recorded_total_turn,recorded_turn_t2.0,468,4.7859,20.9587,121.2123
3,2023,coordinate_total_turn,coordinate_turn_t0,468,45.0000,122.7674,518.7195
4,2023,coordinate_total_turn,coordinate_turn_t1.0,468,8.1301,9.6207,26.5650
5,2023,coordinate_total_turn,coordinate_turn_t2.0,468,28.0725,42.3904,146.4733
6,2023,total_direction_change,recorded_turn_t0,468,4.5722,9.9157,31.6231
7,2023,total_direction_change,coordinate_turn_t0,468,95.7302,134.2296,399.0032
8,2024,recorded_total_turn,recorded_turn_t0,377,6.7206,47.2564,201.2019
9,2024,recorded_total_turn,recorded_turn_t1.0,377,1.4890,2.3969,7.0580



RECONSTRUCTION AUDIT SAVED.


In [49]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
RAW = ROOT / "data/raw/nfl-big-data-bowl-2027/combine_tracking.csv"
AUDIT = ROOT / "reports/data_audit"

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL", "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT", "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT", "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT", "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT", "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

raw = pd.read_csv(
    RAW,
    usecols=[
        "draft_year", "event_id", "nfl_id",
        "entity_type", "time", "drill_name",
        "attempt", "x", "y", "s", "dir"
    ]
)

raw = raw[
    (raw["entity_type"] == "PLAYER") &
    (raw["drill_name"].isin(PRIMARY_DRILLS))
].copy()

raw["time"] = pd.to_datetime(raw["time"], errors="coerce")

for col in ["x", "y", "s", "dir"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

keys = [
    "draft_year", "nfl_id",
    "drill_name", "attempt", "event_id"
]

raw = raw.dropna(subset=keys + ["time", "x", "y", "s", "dir"])
raw = raw.sort_values(keys + ["time"])

def angular_error(a, b):
    return np.abs(((a - b + 180) % 360) - 180)

parts = []

for key, g in raw.groupby(keys, sort=False):

    g = g.sort_values("time")

    dx = np.diff(g["x"].to_numpy())
    dy = np.diff(g["y"].to_numpy())

    coordinate_heading = (
        np.degrees(np.arctan2(dx, dy)) % 360
    )

    recorded_heading = g["dir"].to_numpy()[1:]
    speed = g["s"].to_numpy()[1:]

    error = angular_error(
        coordinate_heading,
        recorded_heading
    )

    tmp = pd.DataFrame({
        "draft_year": key[0],
        "nfl_id": key[1],
        "drill_name": key[2],
        "speed": speed,
        "heading_error": error
    })

    parts.append(tmp)

heading_audit = pd.concat(parts, ignore_index=True)

heading_audit["speed_band"] = pd.cut(
    heading_audit["speed"],
    bins=[-np.inf, 0.5, 1, 2, 3, np.inf],
    labels=["<=0.5", "0.5-1", "1-2", "2-3", ">3"]
)

# Cohort-level speed comparison
speed_summary = (
    heading_audit.groupby(
        ["draft_year", "speed_band"],
        observed=True
    )
    .agg(
        observations=("heading_error", "size"),
        median_error=("heading_error", "median"),
        mean_error=("heading_error", "mean"),
        p90_error=("heading_error", lambda x: x.quantile(.90)),
        p95_error=("heading_error", lambda x: x.quantile(.95))
    )
    .reset_index()
)

print("1. HEADING ERROR BY COHORT AND SPEED")
display(speed_summary.round(3))

# Cohort and drill comparison, using speed > 1
filtered = heading_audit[
    heading_audit["speed"] > 1
].copy()

drill_summary = (
    filtered.groupby(["draft_year", "drill_name"])
    .agg(
        observations=("heading_error", "size"),
        median_error=("heading_error", "median"),
        mean_error=("heading_error", "mean"),
        p90_error=("heading_error", lambda x: x.quantile(.90))
    )
    .reset_index()
)

print("\n2. HEADING ERROR BY COHORT AND DRILL (SPEED > 1)")
display(drill_summary.round(3))

speed_summary.to_csv(
    AUDIT / "heading_error_by_cohort_speed.csv",
    index=False
)

drill_summary.to_csv(
    AUDIT / "heading_error_by_cohort_drill.csv",
    index=False
)

print("\nHEADING COMPARABILITY AUDIT SAVED.")


1. HEADING ERROR BY COHORT AND SPEED


,draft_year,speed_band,observations,median_error,mean_error,p90_error,p95_error
0,2023,<=0.5,853,27.719,43.063,101.057,135.000
1,2023,0.5-1,940,9.135,11.490,23.841,31.435
2,2023,1-2,1905,4.357,5.526,11.889,14.277
3,2023,2-3,1845,2.713,3.245,6.677,8.097
4,2023,>3,29557,1.034,1.299,2.733,3.373
5,2024,<=0.5,1797,32.125,49.968,129.495,159.809
6,2024,0.5-1,772,8.898,11.430,24.127,31.809
7,2024,1-2,1480,4.727,5.731,11.776,14.575
8,2024,2-3,1516,2.886,3.467,7.295,8.559
9,2024,>3,27305,1.046,1.308,2.737,3.419



2. HEADING ERROR BY COHORT AND DRILL (SPEED > 1)


,draft_year,drill_name,observations,median_error,mean_error,p90_error
0,2023,COMEBACK_ROUTE_RIGHT,1872,1.343,2.099,4.854
1,2023,CURL_ROUTE_RIGHT,2167,1.844,2.710,6.150
2,2023,DAGGER_ROUTE_LEFT,2802,1.380,1.741,3.286
3,2023,GAUNTLET_DRILL,10892,0.983,1.347,2.794
4,2023,GO_ROUTE_RIGHT,2900,0.770,1.084,2.002
5,2023,OVER_SHOULDER_ADJUST,2373,1.003,1.384,2.757
6,2023,POST_CORNER_ROUTE_RIGHT,2434,1.051,1.373,2.670
7,2023,RED_ZONE_FADE_RIGHT,1736,1.333,2.072,4.571
8,2023,SLANT_ROUTE_LEFT,1549,1.237,1.749,3.465
9,2023,SLOT_SAIL_ROUTE_RIGHT,2523,1.473,1.742,3.378



HEADING COMPARABILITY AUDIT SAVED.


In [50]:

import numpy as np
import pandas as pd

AUDIT = ROOT / "reports/data_audit"

# -------------------------------------------------------
# 1. Speed distribution by draft cohort
# -------------------------------------------------------

speed_mix = (
    heading_audit.groupby(
        ["draft_year", "speed_band"],
        observed=True
    )
    .agg(
        observations=("heading_error", "size"),
        mean_heading_error=("heading_error", "mean")
    )
    .reset_index()
)

speed_mix["cohort_total"] = (
    speed_mix.groupby("draft_year")["observations"]
    .transform("sum")
)

speed_mix["speed_band_pct"] = (
    speed_mix["observations"] /
    speed_mix["cohort_total"] * 100
)

print("1. SPEED COMPOSITION BY COHORT")
display(speed_mix.round(3))

# -------------------------------------------------------
# 2. Standardize cohort error to a common speed mix
# -------------------------------------------------------

pooled_mix = (
    heading_audit.groupby("speed_band", observed=True)
    .size()
    .rename("pooled_observations")
    .reset_index()
)

pooled_mix["pooled_weight"] = (
    pooled_mix["pooled_observations"] /
    pooled_mix["pooled_observations"].sum()
)

standardized = speed_mix.merge(
    pooled_mix[["speed_band", "pooled_weight"]],
    on="speed_band",
    how="left"
)

standardized["weighted_error"] = (
    standardized["mean_heading_error"] *
    standardized["pooled_weight"]
)

standardized_results = (
    standardized.groupby("draft_year")
    .agg(
        standardized_mean_error=("weighted_error", "sum"),
        observed_mean_error=("mean_heading_error", "mean")
    )
    .reset_index()
)

# Correct observed mean: weight each speed band by its actual
# within-cohort observation share.
actual_weights = speed_mix.copy()
actual_weights["actual_weight"] = (
    actual_weights["observations"] /
    actual_weights["cohort_total"]
)
actual_weights["actual_weighted_error"] = (
    actual_weights["mean_heading_error"] *
    actual_weights["actual_weight"]
)

actual_means = (
    actual_weights.groupby("draft_year")
    ["actual_weighted_error"].sum()
    .rename("actual_overall_mean_error")
    .reset_index()
)

standardized_results = standardized_results.merge(
    actual_means,
    on="draft_year"
)

print("\n2. ACTUAL VS SPEED-STANDARDIZED MEAN ERROR")
display(standardized_results.round(4))

# -------------------------------------------------------
# 3. Focus on observations above speed 1
# -------------------------------------------------------

high_speed = heading_audit[
    heading_audit["speed"] > 1
].copy()

high_speed_summary = (
    high_speed.groupby("draft_year")
    .agg(
        observations=("heading_error", "size"),
        median_error=("heading_error", "median"),
        mean_error=("heading_error", "mean"),
        p90_error=("heading_error", lambda x: x.quantile(.90)),
        p95_error=("heading_error", lambda x: x.quantile(.95))
    )
    .reset_index()
)

print("\n3. HEADING ERROR ABOVE SPEED 1")
display(high_speed_summary.round(4))

# -------------------------------------------------------
# 4. Save outputs
# -------------------------------------------------------

speed_mix.to_csv(
    AUDIT / "heading_speed_composition_by_cohort.csv",
    index=False
)

standardized_results.to_csv(
    AUDIT / "heading_speed_standardized_error.csv",
    index=False
)

high_speed_summary.to_csv(
    AUDIT / "heading_error_above_speed_1.csv",
    index=False
)

print("\nSPEED COMPOSITION AUDIT SAVED.")


1. SPEED COMPOSITION BY COHORT


,draft_year,speed_band,observations,mean_heading_error,cohort_total,speed_band_pct
0,2023,<=0.5,853,43.063,35100,2.430
1,2023,0.5-1,940,11.490,35100,2.678
2,2023,1-2,1905,5.526,35100,5.427
3,2023,2-3,1845,3.245,35100,5.256
4,2023,>3,29557,1.299,35100,84.208
5,2024,<=0.5,1797,49.968,32870,5.467
6,2024,0.5-1,772,11.430,32870,2.349
7,2024,1-2,1480,5.731,32870,4.503
8,2024,2-3,1516,3.467,32870,4.612
9,2024,>3,27305,1.308,32870,83.070



2. ACTUAL VS SPEED-STANDARDIZED MEAN ERROR


,draft_year,standardized_mean_error,observed_mean_error,actual_overall_mean_error
0,2023,3.7325,12.9245,2.9183
1,2024,4.0712,14.3808,4.5045
2,2025,1.9295,6.4793,2.1489



3. HEADING ERROR ABOVE SPEED 1


,draft_year,observations,median_error,mean_error,p90_error,p95_error
0,2023,33307,1.1441,1.6483,3.3786,4.7859
1,2024,30301,1.1373,1.6319,3.3619,4.7483
2,2025,33876,0.4795,0.8988,2.1411,2.7827



SPEED COMPOSITION AUDIT SAVED.
